# Phase 1 – Data understanding
## Step 0: Load all tables

In [1]:
import pandas as pd
import os

DATA = r"C:\Users\start\OneDrive\Desktop\HVIA-TASK3\Cottonil_project\COTTONIL\COTTONIL_Data\02_TABLES"
pd.set_option("display.max_columns", 50)

In [2]:
tables = {}
for f in sorted(os.listdir(DATA)):
    if f.endswith(".csv"):
        tables[f.replace(".csv", "")] = pd.read_csv(os.path.join(DATA, f), low_memory=False)

overview = pd.DataFrame([
    {"table": n, "rows": len(df), "columns": df.shape[1]}
    for n, df in tables.items()
]).sort_values("rows", ascending=False).reset_index(drop=True)
print("Total rows:", f"{overview.rows.sum():,}")
overview

Total rows: 2,931,843


,table,rows,columns
0,sales_order_items,1138536,13
1,sales_orders,540000,13
2,sales_payments,540000,6
3,monthly_stock,142784,7
4,inventory_transfer_items,109614,4
5,purchase_order_items,103334,7
6,delivery_shipments,97660,8
7,customers,85000,6
8,operating_expenses,47000,7
9,purchase_orders,41500,9


## Step 1: `branches`

In [3]:
b = tables["branches"]
print(b.shape)
b.head()

(80, 7)


,branch_id,branch_name_ar,governorate_ar,city_ar,address_ar,retail_location_type,opening_date
0,BR001,الزمالك,القاهرة,الزمالك,شارع 26 يوليو، الزمالك,Point of sale / reseller,9/14/2019
1,BR002,السيدة زينب,القاهرة,السيدة زينب,229 شارع بورسعيد، السيدة زينب,Point of sale / reseller,2/19/2019
2,BR003,القطامية داون تاون,القاهرة,القطامية,شارع التسعين، مول داون تاون، القطامية,Retail branch,2/12/2020
3,BR004,الرحاب مول,القاهرة,الرحاب,المرحلة الثالثة، مول الرحاب,Point of sale / reseller,8/14/2019
4,BR005,المعادي جراند مول,القاهرة,المعادي,شارع 250، جراند مول، المعادي,Retail branch,11/16/2018


In [4]:
print("Empty cells:", b.isna().sum().sum())
print("Duplicate branch_id:", b.branch_id.duplicated().sum())
print("Unique governorates:", b.governorate_ar.nunique(), "| Unique cities:", b.city_ar.nunique())

Empty cells: 0
Duplicate branch_id: 0
Unique governorates: 21 | Unique cities: 65


In [5]:
print(b.retail_location_type.value_counts(), "\n")
b.governorate_ar.value_counts().to_frame("branches")

retail_location_type
Retail branch               52
Point of sale / reseller    28
Name: count, dtype: int64 



,branches
governorate_ar,
القاهرة,21
الإسكندرية,10
الجيزة,6
الدقهلية,4
البحيرة,4
الغربية,4
المنيا,4
الشرقية,3
المنوفية,3


In [6]:
# branches inside malls
mall = b[b.branch_name_ar.str.contains("مول") | b.address_ar.str.contains("مول")]
print("Branches inside malls:", len(mall))
mall[["branch_id", "branch_name_ar", "governorate_ar", "retail_location_type"]]

Branches inside malls: 8


,branch_id,branch_name_ar,governorate_ar,retail_location_type
2,BR003,القطامية داون تاون,القاهرة,Retail branch
3,BR004,الرحاب مول,القاهرة,Point of sale / reseller
4,BR005,المعادي جراند مول,القاهرة,Retail branch
5,BR006,فيوتشر مول,القاهرة,Retail branch
18,BR019,كايرو فيستيفال,القاهرة,Retail branch
50,BR051,الأردنية,الشرقية,Retail branch
69,BR070,مول الصفوة,دمياط,Retail branch
79,BR080,الجامعة,أسيوط,Retail branch


In [7]:
b["opening_date"] = pd.to_datetime(b.opening_date, format="%m/%d/%Y")
first_sale = pd.to_datetime(tables["sales_orders"].groupby("branch_id").order_date.min())

check = b[["branch_id", "branch_name_ar", "opening_date"]].copy()
check["first_sale"] = check.branch_id.map(first_sale)
check["sold_before_opening_days"] = (check.opening_date - check.first_sale).dt.days

print("Opening dates range:", check.opening_date.min().date(), "->", check.opening_date.max().date())
print("First sale range:   ", check.first_sale.min().date(), "->", check.first_sale.max().date())
print("Branches that sold BEFORE their opening date:", (check.sold_before_opening_days > 0).sum(), "of", len(check))
check.sort_values("sold_before_opening_days", ascending=False).head()

Opening dates range: 2018-11-06 -> 2020-08-29
First sale range:    2018-10-19 -> 2018-10-22
Branches that sold BEFORE their opening date: 80 of 80


,branch_id,branch_name_ar,opening_date,first_sale,sold_before_opening_days
16,BR017,زهراء المعادي,2020-08-29,2018-10-19,680
40,BR041,طلخا,2020-08-26,2018-10-19,677
77,BR078,شرق البندر,2020-08-10,2018-10-19,661
11,BR012,البطراوي,2020-08-07,2018-10-20,657
55,BR056,سقالة,2020-07-26,2018-10-19,646


In [8]:
# orders that came BEFORE their branch opened, by sales channel
so = tables["sales_orders"]
x = so.merge(b[["branch_id", "opening_date", "retail_location_type"]], on="branch_id")
before = x[pd.to_datetime(x.order_date) < x.opening_date]
print("Orders before their branch opening date:", len(before), f"({len(before) / len(so):.1%})")
before.sales_channel.value_counts().to_frame("orders before opening")

Orders before their branch opening date: 38040 (7.0%)


,orders before opening
sales_channel,
Retail POS,27415
E-commerce,6783
Wholesale / reseller,3842


In [9]:
print(b[b.branch_name_ar.duplicated(keep=False)][["branch_id", "branch_name_ar", "governorate_ar", "address_ar"]], "\n")

for t, col in [("sales_orders", "branch_id"), ("employees", "branch_id"), ("inventory_transfers", "destination_branch_id")]:
    missing = (~tables[t][col].isin(b.branch_id)).sum()
    print(f"{t}.{col}: {tables[t][col].nunique()} branches used, {missing} codes not found in branches")

dd = tables["data_dictionary"]
print("\nColumns in dictionary but NOT in file:",
      set(dd[dd.table_name == "branches.csv"].field_name) - set(tables["branches"].columns))

   branch_id branch_name_ar governorate_ar                address_ar
41     BR042      سعد زغلول    الإسماعيلية        190 شارع سعد زغلول
56     BR057      سعد زغلول         الفيوم  ممر أبو عميرة، سعد زغلول 

sales_orders.branch_id: 80 branches used, 0 codes not found in branches
employees.branch_id: 80 branches used, 0 codes not found in branches
inventory_transfers.destination_branch_id: 80 branches used, 0 codes not found in branches

Columns in dictionary but NOT in file: {'source_url'}


In [10]:
# does a branch's place or type change how much it sells?
bs = b.set_index("branch_id")
bs["store_orders"] = so[so.sales_channel == "Retail POS"].groupby("branch_id").size()
bs["in_mall"] = bs.index.isin(mall.branch_id)
print("In-store orders per branch: min", bs.store_orders.min(), "| max", bs.store_orders.max())
print("Average by type:", bs.groupby("retail_location_type").store_orders.mean().round(0).to_dict())
print("Average in mall vs not:", bs.groupby("in_mall").store_orders.mean().round(0).to_dict(), "\n")
bs.sort_values("store_orders")[["branch_name_ar", "governorate_ar", "retail_location_type", "store_orders"]].iloc[[0, 1, -2, -1]]

In-store orders per branch: min 4699 | max 5011
Average by type: {'Point of sale / reseller': 4863.0, 'Retail branch': 4846.0}
Average in mall vs not: {False: 4855.0, True: 4824.0} 



,branch_name_ar,governorate_ar,retail_location_type,store_orders
branch_id,,,,
BR067,بني سويف الجديدة,بني سويف,Retail branch,4699
BR020,طلعت حرب,القاهرة,Retail branch,4723
BR003,القطامية داون تاون,القاهرة,Retail branch,4987
BR056,سقالة,البحر الأحمر,Point of sale / reseller,5011


In [11]:
# company branches vs reseller points: same mix of channels?
online = so[so.sales_channel == "E-commerce"].groupby("branch_id").size()
print("Branches with online orders:", online.size, "| per branch: min", online.min(), "| max", online.max())
pd.crosstab(x.retail_location_type, x.sales_channel, normalize="index").round(3)

Branches with online orders: 80 | per branch: min 1148 | max 1354


sales_channel,E-commerce,Retail POS,Wholesale / reseller
retail_location_type,,,
Point of sale / reseller,0.182,0.719,0.099
Retail branch,0.180,0.719,0.101


In [12]:
# where does a branch code appear in other tables?
for name, df in tables.items():
    cols = [col for col in df.columns if "branch" in col or col == "stock_location_id"]
    if cols and name != "branches":
        print(f"{name:20s}", cols)

employees            ['branch_id']
inventory_transfers  ['destination_branch_id']
monthly_stock        ['stock_location_id']
operating_expenses   ['branch_or_department_id']
sales_order_items    ['branch_id']
sales_orders         ['branch_id']


## Step 2: `customers`

In [13]:
c = tables["customers"]
print(c.shape)
c.head()

(85000, 6)


,customer_id,customer_segment,customer_name_ar,governorate_ar,joined_date,age_band
0,C0000001,Retail,كريم حمادة,دمياط,10/19/2018,55+
1,C0000002,Retail,مريم خليل,أسوان,10/19/2018,18-24
2,C0000003,Retail,خالد محمد,القاهرة,10/19/2018,45-54
3,C0000004,Retail,أحمد منصور,المنوفية,10/19/2018,35-44
4,C0000005,Retail,رانيا إبراهيم,المنوفية,10/19/2018,18-24


In [14]:
print("Empty cells:", c.isna().sum().sum())
print("Duplicate customer_id:", c.customer_id.duplicated().sum())
print("Columns in dictionary but NOT in file:",
      set(dd[dd.table_name == "customers.csv"].field_name) - set(c.columns))
print("Unique names:", c.customer_name_ar.nunique(), "for", len(c), "customers")
c.customer_name_ar.value_counts().head(3).to_frame("customers with this name")

Empty cells: 0
Duplicate customer_id: 0
Columns in dictionary but NOT in file: {'loyalty_tier'}
Unique names: 2233 for 85000 customers


,customers with this name
customer_name_ar,
يارا الشرقاوي,184
عبدالله علي,184
منى النجار,181


In [15]:
print(c.customer_segment.value_counts(), "\n")
print(c.age_band.value_counts().sort_index(), "\n")
gov = c.governorate_ar.value_counts()
print("Customers per governorate: min", gov.min(), "| max", gov.max(), "| governorates", len(gov), "\n")
c["joined_date"] = pd.to_datetime(c.joined_date, format="%m/%d/%Y")
c.joined_date.dt.year.value_counts().sort_index().to_frame("new customers")

customer_segment
Retail          83334
B2B reseller     1666
Name: count, dtype: int64 

age_band
18-24    16952
25-34    17058
35-44    16895
45-54    16945
55+      17150
Name: count, dtype: int64 

Customers per governorate: min 3956 | max 4134 | governorates 21 



,new customers
joined_date,
2018,2153
2019,10618
2020,10647
2021,10617
2022,10618
2023,10618
2024,10647
2025,10617
2026,8465


In [16]:
so = tables["sales_orders"]
print("Orders with NO customer_id:", so.customer_id.isna().sum(), f"({so.customer_id.isna().mean():.1%})")
print("customer_id in orders not found in customers:", (~so.customer_id.dropna().isin(c.customer_id)).sum())

buyers = so.dropna(subset=["customer_id"])
print("Customers who bought:", buyers.customer_id.nunique(),
      "| Registered but never bought:", (~c.customer_id.isin(buyers.customer_id)).sum())

first = pd.to_datetime(buyers.groupby("customer_id").order_date.min())
gap = (first - c.set_index("customer_id").joined_date.reindex(first.index)).dt.days
print("First order BEFORE joining:", (gap < 0).sum(), "| median days from joining to first order:", gap.median())
print("Orders per customer:", buyers.groupby("customer_id").size().describe().round(1).to_dict())

Orders with NO customer_id: 102961 (19.1%)
customer_id in orders not found in customers: 0
Customers who bought: 74660 | Registered but never bought: 10340
First order BEFORE joining: 0 | median days from joining to first order: 139.0
Orders per customer: {'count': 74660.0, 'mean': 5.9, 'std': 4.4, 'min': 1.0, '25%': 2.0, '50%': 5.0, '75%': 8.0, 'max': 43.0}


In [17]:
# is the missing customer only walk-in shoppers? check every channel
print(so.customer_id.isna().groupby(so.sales_channel).mean().round(3).to_frame("share without customer"), "\n")
online_unknown = so[(so.sales_channel == "E-commerce") & so.customer_id.isna()]
print("Online orders without a customer:", len(online_unknown),
      "| shipped to someone:", online_unknown.sales_order_id.isin(tables["delivery_shipments"].sales_order_id).sum())

                      share without customer
sales_channel                               
E-commerce                             0.189
Retail POS                             0.191
Wholesale / reseller                   0.193 

Online orders without a customer: 18430 | shipped to someone: 18430


In [18]:
seg = buyers.merge(c[["customer_id", "customer_segment"]], on="customer_id")
print(pd.crosstab(seg.sales_channel, seg.customer_segment), "\n")

br = tables["branches"][["branch_id", "governorate_ar"]]
m = buyers.merge(c[["customer_id", "governorate_ar"]], on="customer_id") \
          .merge(br, on="branch_id", suffixes=("_customer", "_branch"))
store = m[m.sales_channel == "Retail POS"]
print("In-store orders where customer lives in the SAME governorate as the branch:",
      f"{(store.governorate_ar_customer == store.governorate_ar_branch).mean():.1%}",
      "(random chance with 21 governorates = 4.8%)")

customer_segment      B2B reseller  Retail
sales_channel                             
E-commerce                    1503   77727
Retail POS                    6233  307885
Wholesale / reseller           851   42840 

In-store orders where customer lives in the SAME governorate as the branch: 4.8% (random chance with 21 governorates = 4.8%)


In [19]:
# how many different branches does one customer visit?
visits = buyers[buyers.sales_channel == "Retail POS"].merge(br, on="branch_id")
per_customer = visits.groupby("customer_id").agg(visits=("sales_order_id", "size"),
                                                 branches=("branch_id", "nunique"),
                                                 governorates=("governorate_ar", "nunique"))
regular = per_customer[per_customer.visits >= 5]
print("Customers with 5+ store visits:", len(regular))
regular.mean().round(1).to_frame("average per customer")

Customers with 5+ store visits: 27707


,average per customer
visits,7.7
branches,7.3
governorates,5.6


In [20]:
# where does a customer lead? customer_id -> sales_orders -> sales_order_id -> other tables
for name, df in tables.items():
    cols = [col for col in df.columns if col in ("customer_id", "sales_order_id")]
    if cols and name != "customers":
        print(f"{name:20s}", cols)

delivery_shipments   ['sales_order_id']
sales_order_items    ['sales_order_id']
sales_orders         ['sales_order_id', 'customer_id']
sales_payments       ['sales_order_id']
sales_returns        ['sales_order_id']


## Step 3: `products_catalog`

In [21]:
p = tables["products_catalog"]
print(p.shape)
p.head()

(139, 6)


,product_id,product_name_ar,category_en,audience,listed_price_EGP,source_url
0,PR0001,3 * شراب رجالي كلاسيك-لايكرا 1,Socks,Men,290,https://cottonilegypt.com/collections/all?page=1
1,PR0002,3 * شراب رجالي كلاسيك-لايكرا 2,Socks,Men,290,https://cottonilegypt.com/collections/all?page=1
2,PR0003,3 * شراب رجالي كلاسيك-لايكرا 3,Socks,Men,290,https://cottonilegypt.com/collections/all?page=1
3,PR0004,3 * شراب رجالي كلاسيك-لايكرا 4,Socks,Men,290,https://cottonilegypt.com/collections/all?page=1
4,PR0005,3 * شراب رجالي كلاسيك-لايكرا 5,Socks,Men,290,https://cottonilegypt.com/collections/all?page=1


In [22]:
print("Empty cells:", p.isna().sum().sum(),
      "| duplicate ids:", p.product_id.duplicated().sum(),
      "| duplicate names:", p.product_name_ar.duplicated().sum())
p.category_en.value_counts().to_frame("products")

Empty cells: 0 | duplicate ids: 0 | duplicate names: 0


,products
category_en,
Socks,76
Men tops,9
Kids underwear,8
Women tops,7
Men boxers,7
Women bras,6
Men underwear,5
Home textiles,4
Boys pajamas,4


In [23]:
# who is each category for? (look at Women bras -> Unisex)
pd.crosstab(p.category_en, p.audience)

audience,Boys,Girls,Kids,Men,Unisex,Women
category_en,,,,,,
Accessories,0,0,0,0,1,0
Boys pajamas,4,0,0,0,0,0
Girls pajamas,0,4,0,0,0,0
Girls underwear,0,0,0,0,1,0
Home textiles,0,0,0,0,4,0
Kids underwear,0,0,8,0,0,0
Men bottoms,0,0,0,3,0,0
Men boxers,0,0,0,2,5,0
Men tops,0,0,0,7,2,0


In [24]:
# price per category (price is for the whole pack, 2026 website price)
p.groupby("category_en").listed_price_EGP.agg(["count", "min", "median", "max"]).sort_values("median")

,count,min,median,max
category_en,,,,
Kids underwear,8,190,190.0,190
Women tops,7,190,190.0,550
Women bottoms,1,255,255.0,255
Socks,76,220,290.0,485
Men bottoms,3,310,310.0,395
Accessories,1,435,435.0,435
Girls pajamas,4,500,500.0,500
Boys pajamas,4,500,500.0,500
Women bras,6,225,505.0,505


In [25]:
# near-duplicate products: same name with a number at the end
base = p.product_name_ar.str.replace(r"[\d\-]+\s*$", "", regex=True).str.strip()
base.value_counts().head(8).to_frame("copies")

,copies
product_name_ar,
3 * شراب رجالي كلاسيك-لايكرا,57
بكيني كاندي مطبوع 2 قطعة توينز,8
ايزي توب قطن,4
بكيني رجالي مارسريزية,4
بيجامة صيفي اولادي - كود,4
بيجامة صيفي بناتي - كود,4
بيجامة شتوي حريمي كود,2
3 طقم بناتى هوت شورت,1


## Step 4: `sku_variants`

In [26]:
v = tables["sku_variants"]
print(v.shape)
v.head(10)

(695, 5)


,sku_id,product_id,size,color_ar,price_multiplier
0,SKU000001,PR0001,Free,أبيض,0.950
1,SKU000002,PR0001,M,أسود,0.975
2,SKU000003,PR0001,L,كحلي,1.000
3,SKU000004,PR0001,XL,رمادي,1.025
4,SKU000005,PR0001,Free,بيج,1.050
5,SKU000006,PR0002,M,رمادي,0.950
6,SKU000007,PR0002,L,بيج,0.975
7,SKU000008,PR0002,XL,أزرق,1.000
8,SKU000009,PR0002,Free,نبيتي,1.025
9,SKU000010,PR0002,M,وردي,1.050


In [27]:
print("Empty cells:", v.isna().sum().sum(), "| duplicate sku_id:", v.sku_id.duplicated().sum())
print("Columns in dictionary but NOT in file:", set(dd[dd.table_name == "sku_variants.csv"].field_name) - set(v.columns))
print("SKUs per product:", v.groupby("product_id").size().value_counts().to_dict(),
      "| products with no SKU:", (~p.product_id.isin(v.product_id)).sum())
print("Sizes:", v["size"].value_counts().to_dict())
print("Colours:", v.color_ar.value_counts().to_dict())
v.price_multiplier.value_counts().sort_index().to_frame("SKUs")

Empty cells: 0 | duplicate sku_id: 0
Columns in dictionary but NOT in file: set()
SKUs per product: {5: 139} | products with no SKU: 0
Sizes: {'XL': 142, 'L': 141, 'M': 136, 'Free': 113, '2XL': 28, '4XL': 28, '3XL': 27, '3': 14, '6': 14, '2': 13, '4': 13, '5': 13, '7': 13}
Colours: {'أبيض': 87, 'أسود': 87, 'كحلي': 87, 'رمادي': 87, 'بيج': 87, 'نبيتي': 87, 'وردي': 87, 'أزرق': 86}


,SKUs
price_multiplier,
0.950,139
0.975,139
1.000,139
1.025,139
1.050,139


In [28]:
# 5 SKUs per product, each one a different colour -> every colour exists in ONE size only
print("Products with the same colour twice:", v.groupby("product_id").color_ar.apply(lambda s: s.duplicated().any()).sum())
print("Products with the same size twice:", v.groupby("product_id")["size"].apply(lambda s: s.duplicated().any()).sum())
v[v.product_id.isin(["PR0001", "PR0060"])]

Products with the same colour twice: 0
Products with the same size twice: 89


,sku_id,product_id,size,color_ar,price_multiplier
0,SKU000001,PR0001,Free,أبيض,0.950
1,SKU000002,PR0001,M,أسود,0.975
2,SKU000003,PR0001,L,كحلي,1.000
3,SKU000004,PR0001,XL,رمادي,1.025
4,SKU000005,PR0001,Free,بيج,1.050
295,SKU000296,PR0060,4XL,أسود,0.950
296,SKU000297,PR0060,M,كحلي,0.975
297,SKU000298,PR0060,L,رمادي,1.000
298,SKU000299,PR0060,XL,بيج,1.025
299,SKU000300,PR0060,2XL,أزرق,1.050


In [29]:
# price_multiplier follows the SKU's position inside its product (1st, 2nd, ...), not its size
v["position"] = v.groupby("product_id").cumcount() + 1
pd.crosstab(v.position, v.price_multiplier)

price_multiplier,0.95,0.975,1.0,1.025,1.05
position,,,,,
1,139,0,0,0,0
2,0,139,0,0,0
3,0,0,139,0,0
4,0,0,0,139,0
5,0,0,0,0,139


In [30]:
# odd sizes: a girls' product and towels
vp = v.merge(p[["product_id", "product_name_ar", "category_en"]], on="product_id")
vp[vp.category_en.isin(["Girls underwear", "Home textiles"])].groupby(["product_id", "product_name_ar"])["size"].apply(" ".join).to_frame("sizes")

,,sizes
product_id,product_name_ar,
PR0058,3 طقم بناتى هوت شورت,2XL 3XL 4XL M L
PR0062,3 قطع فوط مقاس 100*50,L XL Free L XL
PR0102,بشكير احرام قطونيل,XL Free L XL Free
PR0103,بشكير قطونيل كبير ( 90x160 ),Free L XL Free L
PR0104,بشكير قطونيل وسط ( 70x140 ),L XL Free L XL


In [31]:
# products whose NAME says one colour, but the SKUs have 5 colours
colour_word = r"(?:^|[\s\-])(?:ابيض|أبيض|اسود|أسود|بيج|ازرق|أزرق|كحلي|رمادي|نبيتي|وردي)(?:$|[\s\-])"
named = p[p.product_name_ar.str.contains(colour_word, regex=True) & ~p.product_name_ar.str.contains("الوان")]
print("Products with one colour in their name:", len(named))
vp[vp.product_id.isin(named.product_id)].groupby("product_name_ar").color_ar.apply(" - ".join).to_frame("SKU colours")

Products with one colour in their name: 14


,SKU colours
product_name_ar,
3* شراب ميني سوكيت أبيض,أزرق - نبيتي - وردي - أبيض - أسود
3*شراب قدم أبيض,أبيض - أسود - كحلي - رمادي - بيج
3*شراب قدم أزرق فاتح,أسود - كحلي - رمادي - بيج - أزرق
3*شراب قدم ازرق غامق,وردي - أبيض - أسود - كحلي - رمادي
3*شراب قدم اسود,كحلي - رمادي - بيج - أزرق - نبيتي
3*شراب قدم بيج,أبيض - أسود - كحلي - رمادي - بيج
3*شراب ميني سوكيت أزرق غامق,وردي - أبيض - أسود - كحلي - رمادي
3*شراب ميني سوكيت أسود,كحلي - رمادي - بيج - أزرق - نبيتي
برا بدن دانتیل بوش اب-ابيض,أزرق - نبيتي - وردي - أبيض - أسود


In [32]:
# is the price_multiplier really used in the selling price? compare each SKU with its product in the same month
si = tables["sales_order_items"]
s = si.merge(v[["sku_id", "price_multiplier"]], on="sku_id")
month = s.order_date.str[:7]
s["price_vs_product"] = s.unit_price_EGP / s.groupby([s.product_id, month]).unit_price_EGP.transform("mean")
v["units"] = v.sku_id.map(si.groupby("sku_id").quantity.sum())
pd.DataFrame({"selling price vs product": s.groupby("price_multiplier").price_vs_product.mean().round(3),
              "units sold per SKU": v.groupby("price_multiplier").units.mean().round(0)})

,selling price vs product,units sold per SKU
price_multiplier,,
0.950,0.951,4896.0
0.975,0.975,4889.0
1.000,1.000,4898.0
1.025,1.025,4885.0
1.050,1.049,4883.0


In [33]:
# inside the same product, do size or colour change how much sells?
v["vs_product"] = v.units / v.groupby("product_id").units.transform("mean")
print("Units vs own product, by size:", v.groupby("size").vs_product.mean().round(2).to_dict())
print("Units vs own product, by colour:", v.groupby("color_ar").vs_product.mean().round(2).to_dict(), "\n")
clothes = v[v.product_id.isin(v[v["size"].isin(["2XL", "3XL", "4XL"])].product_id)]
print("Clothes with sizes M to 4XL:", clothes.product_id.nunique(), "products")
clothes.groupby("size").units.mean().round(0).to_frame("units sold per SKU")

Units vs own product, by size: {'2': 1.0, '2XL': 1.01, '3': 1.02, '3XL': 1.0, '4': 1.0, '4XL': 1.0, '5': 0.98, '6': 1.01, '7': 0.98, 'Free': 1.0, 'L': 1.0, 'M': 0.99, 'XL': 1.0}
Units vs own product, by colour: {'أبيض': 0.99, 'أزرق': 0.99, 'أسود': 1.0, 'بيج': 1.0, 'رمادي': 1.0, 'كحلي': 1.02, 'نبيتي': 1.0, 'وردي': 1.0} 

Clothes with sizes M to 4XL: 34 products


,units sold per SKU
size,
2XL,8942.0
3XL,8998.0
4XL,9009.0
L,9115.0
M,8783.0
XL,9080.0


In [34]:
# where does sku_id appear?
for name, df in tables.items():
    if "sku_id" in df.columns and name != "sku_variants":
        print(f"{name:26s}", df.sku_id.nunique(), "SKUs used |", (~df.sku_id.isin(v.sku_id)).sum(), "unknown codes")
print("\nmonthly_stock is kept by:", [col for col in tables["monthly_stock"].columns if "product" in col or "sku" in col])

inventory_transfer_items   695 SKUs used | 0 unknown codes
purchase_order_items       695 SKUs used | 0 unknown codes
sales_order_items          695 SKUs used | 0 unknown codes
sales_returns              695 SKUs used | 0 unknown codes

monthly_stock is kept by: ['product_category']


## Step 5: `warehouses`

In [35]:
w = tables["warehouses"]
print(w.shape)
w

(12, 5)


,warehouse_id,warehouse_label,governorate_ar,warehouse_kind,capacity_units
0,WH01,مخزن مركزي القاهرة,القاهرة,Central DC,262541
1,WH02,مخزن التوزيع الإسكندرية,الإسكندرية,Regional DC,119583
2,WH03,مخزن التوزيع الجيزة,الجيزة,Regional DC,143948
3,WH04,مخزن التوزيع الدلتا,الدقهلية,Regional DC,114899
4,WH05,مخزن التوزيع الصعيد,المنيا,Regional DC,96050
5,WH06,مخزن القطن الخام,القاهرة,Raw materials,138566
6,WH07,مخزن المنتج التام,القاهرة,Finished goods,307157
7,WH08,مخزن المرتجعات,القاهرة,Returns,332021
8,WH09,مخزن توزيع الشرقية,الشرقية,Regional DC,159485
9,WH10,مخزن توزيع البحيرة,البحيرة,Regional DC,111800


In [36]:
print("Empty cells:", w.isna().sum().sum(), "| duplicate warehouse_id:", w.warehouse_id.duplicated().sum())
print("Columns in dictionary but NOT in file:", set(dd[dd.table_name == "warehouses.csv"].field_name) - set(w.columns))
print("Kinds:", w.warehouse_kind.value_counts().to_dict())
print("Total capacity:", w.capacity_units.sum())

Empty cells: 0 | duplicate warehouse_id: 0
Columns in dictionary but NOT in file: set()
Kinds: {'Regional DC': 8, 'Central DC': 1, 'Raw materials': 1, 'Finished goods': 1, 'Returns': 1}
Total capacity: 2418197


In [37]:
# capacity vs how many branches each region has
region = {"القاهرة": "Cairo & Giza", "الجيزة": "Cairo & Giza", "الإسكندرية": "Alexandria",
          "الإسماعيلية": "Canal", "بورسعيد": "Canal", "المنيا": "Upper Egypt", "الفيوم": "Upper Egypt",
          "بني سويف": "Upper Egypt", "أسيوط": "Upper Egypt", "سوهاج": "Upper Egypt", "قنا": "Upper Egypt", "أسوان": "Upper Egypt"}
print(b.governorate_ar.map(region).fillna("Delta & others").value_counts().to_dict(), "\n")
w.sort_values("capacity_units")[["warehouse_id", "warehouse_label", "warehouse_kind", "capacity_units"]]

{'Cairo & Giza': 27, 'Delta & others': 24, 'Upper Egypt': 15, 'Alexandria': 10, 'Canal': 4} 



,warehouse_id,warehouse_label,warehouse_kind,capacity_units
4,WH05,مخزن التوزيع الصعيد,Regional DC,96050
9,WH10,مخزن توزيع البحيرة,Regional DC,111800
3,WH04,مخزن التوزيع الدلتا,Regional DC,114899
1,WH02,مخزن التوزيع الإسكندرية,Regional DC,119583
5,WH06,مخزن القطن الخام,Raw materials,138566
2,WH03,مخزن التوزيع الجيزة,Regional DC,143948
8,WH09,مخزن توزيع الشرقية,Regional DC,159485
0,WH01,مخزن مركزي القاهرة,Central DC,262541
11,WH12,مخزن توزيع القناة,Regional DC,299463
6,WH07,مخزن المنتج التام,Finished goods,307157


In [38]:
# what does each warehouse send to branches?
it = tables["inventory_transfers"].copy()
it["units"] = it.transfer_id.map(tables["inventory_transfer_items"].groupby("transfer_id").quantity_units.sum())
sent = it.groupby("source_warehouse_id").units.sum()
w.set_index("warehouse_id")[["warehouse_kind", "capacity_units"]].assign(units_sent=sent, share=(sent / sent.sum()).round(3))

,warehouse_kind,capacity_units,units_sent,share
warehouse_id,,,,
WH01,Central DC,262541,1018018,0.080
WH02,Regional DC,119583,1070462,0.084
WH03,Regional DC,143948,1065330,0.084
WH04,Regional DC,114899,1057948,0.083
WH05,Regional DC,96050,1048309,0.083
WH06,Raw materials,138566,1039983,0.082
WH07,Finished goods,307157,1072710,0.085
WH08,Returns,332021,1104403,0.087
WH09,Regional DC,159485,1075537,0.085


In [39]:
# what is stored in the raw-cotton warehouse? (monthly_stock, latest month)
ms = tables["monthly_stock"]
ms[(ms.stock_location_id == "WH06") & (ms.year_month == ms.year_month.max())][["product_category", "closing_units"]]

,product_category,closing_units
142672,Accessories,3786
142673,Boys pajamas,12289
142674,Girls pajamas,6289
142675,Girls underwear,8744
142676,Home textiles,788
142677,Kids underwear,10004
142678,Men bottoms,829
142679,Men boxers,4720
142680,Men tops,1309
142681,Men underwear,1913


In [40]:
# who receives purchase orders and imports?
po = tables["purchase_orders"]
po_units = tables["purchase_order_items"].groupby("po_id").quantity.sum()
received = po.assign(units=po.po_id.map(po_units)).groupby("receiving_warehouse_id").agg(
    purchase_orders=("po_id", "size"), units_ordered=("units", "sum"))
print(w.set_index("warehouse_id")[["warehouse_kind"]].join(received).fillna(0).assign(units_sent=sent), "\n")
imp = tables["import_shipments"]
pd.crosstab(imp.destination_warehouse_id.map(w.set_index("warehouse_id").warehouse_kind), imp.commodity_class)

              warehouse_kind  purchase_orders  units_ordered  units_sent
warehouse_id                                                            
WH01              Central DC           3790.0     30864257.0     1018018
WH02             Regional DC           3672.0     29595251.0     1070462
WH03             Regional DC           3743.0     30550375.0     1065330
WH04             Regional DC           3672.0     30014698.0     1057948
WH05             Regional DC           3684.0     29868587.0     1048309
WH06           Raw materials           3756.0     30607786.0     1039983
WH07          Finished goods           3847.0     31361334.0     1072710
WH08                 Returns           3732.0     30728194.0     1104403
WH09             Regional DC           3899.0     32013287.0     1075537
WH10             Regional DC           3811.0     31593785.0     1040445
WH11             Regional DC           3894.0     32254613.0     1062463
WH12             Regional DC              0.0      

commodity_class,Accessories,Cotton fabric,Industrial machinery,Packaging materials
destination_warehouse_id,,,,
Central DC,83,77,86,80
Finished goods,103,84,83,95
Raw materials,78,74,74,95
Regional DC,683,741,690,707
Returns,85,96,89,97


In [41]:
# does a branch get goods from the nearest warehouse?
t = it.merge(b[["branch_id", "governorate_ar"]], left_on="destination_branch_id", right_on="branch_id")
t["warehouse_gov"] = t.source_warehouse_id.map(w.set_index("warehouse_id").governorate_ar)
print("Warehouses each branch received from:", t.groupby("branch_id").source_warehouse_id.nunique().unique())
same = (t.warehouse_gov == t.governorate_ar).mean()
if_random = (t.governorate_ar.map(w.governorate_ar.value_counts()).fillna(0) / len(w)).mean()
print(f"Transfers from a warehouse in the branch's own governorate: {same:.1%} (if random: {if_random:.1%})")
t[t.governorate_ar == "الإسكندرية"].source_warehouse_id.value_counts(normalize=True).round(3).head(4).to_frame("Alexandria branches")

Warehouses each branch received from: [12]
Transfers from a warehouse in the branch's own governorate: 12.8% (if random: 12.5%)


,Alexandria branches
source_warehouse_id,
WH02,0.096
WH03,0.086
WH09,0.086
WH05,0.085


In [42]:
# does capacity matter? average stock vs capacity (monthly_stock)
stock = ms[ms.stock_location_type == "Warehouse"].groupby(["stock_location_id", "year_month"]).closing_units.sum()
cap = w.set_index("warehouse_id").capacity_units
s = stock.groupby(level=0).mean().round(0).to_frame("avg_stock").assign(capacity=cap)
s["full_%"] = (s.avg_stock / s.capacity * 100).round(0)
s["months_over_capacity"] = stock.groupby(level=0).apply(lambda x: (x.values > cap[x.name]).sum())
s.sort_values("capacity")

,avg_stock,capacity,full_%,months_over_capacity
stock_location_id,,,,
WH05,120457.0,96050,125.0,88
WH10,121945.0,111800,109.0,72
WH04,121538.0,114899,106.0,62
WH02,122644.0,119583,103.0,58
WH06,119585.0,138566,86.0,11
WH03,122211.0,143948,85.0,4
WH09,122859.0,159485,77.0,2
WH01,122430.0,262541,47.0,0
WH12,121198.0,299463,40.0,0


In [43]:
# warehouse expenses vs size (operating_expenses, all years)
ox = tables["operating_expenses"]
cost = ox[ox.branch_or_department_id.str.startswith("WH")].groupby("branch_or_department_id").amount_EGP.sum()
(cost / 1e6).round(2).to_frame("expenses (M EGP)").assign(capacity=cap).sort_values("capacity")

,expenses (M EGP),capacity
branch_or_department_id,,
WH05,2.30,96050
WH10,2.19,111800
WH04,2.37,114899
WH02,2.20,119583
WH06,2.12,138566
WH03,2.33,143948
WH09,2.11,159485
WH01,2.32,262541
WH12,2.35,299463


## Step 6: `suppliers`

In [44]:
sup = tables["suppliers"]
print(sup.shape)
sup.head(12)

(48, 5)


,supplier_id,supplier_name_ar,material_type,city_country,payment_terms_days
0,SUP001,مورد خامات رقم 001,Packaging,بورسعيد، مصر,45
1,SUP002,مورد خامات رقم 002,Accessories,بورسعيد، مصر,90
2,SUP003,مورد خامات رقم 003,Finished garments,الجيزة، مصر,45
3,SUP004,مورد خامات رقم 004,Elastic & Lycra,القاهرة، مصر,60
4,SUP005,مورد خامات رقم 005,Labels,القاهرة، مصر,45
5,SUP006,مورد خامات رقم 006,Machinery services,القاهرة، مصر,60
6,SUP007,مورد خامات رقم 007,Knitted fabric,القاهرة، مصر,45
7,SUP008,مورد خامات رقم 008,Dyes,بورسعيد، مصر,90
8,SUP009,مورد خامات رقم 009,Logistics,الإسكندرية، مصر,90
9,SUP010,مورد خامات رقم 010,Cotton yarn,دمياط، مصر,15


In [45]:
print("Empty cells:", sup.isna().sum().sum(), "| duplicate supplier_id:", sup.supplier_id.duplicated().sum())
print("Columns in dictionary but NOT in file:", set(dd[dd.table_name == "suppliers.csv"].field_name) - set(sup.columns))
print("Names that start with 'مورد خامات رقم':", sup.supplier_name_ar.str.startswith("مورد خامات رقم").sum(), "of", len(sup))
print("Material types:", sup.material_type.value_counts().to_dict())
print("Cities:", sup.city_country.value_counts().to_dict())
print("Payment terms (days):", sup.payment_terms_days.value_counts().sort_index().to_dict())

Empty cells: 0 | duplicate supplier_id: 0
Columns in dictionary but NOT in file: set()
Names that start with 'مورد خامات رقم': 48 of 48
Material types: {'Packaging': 5, 'Accessories': 5, 'Finished garments': 5, 'Elastic & Lycra': 5, 'Labels': 5, 'Machinery services': 5, 'Knitted fabric': 5, 'Dyes': 5, 'Logistics': 4, 'Cotton yarn': 4}
Cities: {'بورسعيد، مصر': 12, 'الإسكندرية، مصر': 11, 'القاهرة، مصر': 9, 'الجيزة، مصر': 7, 'المحلة الكبرى، مصر': 5, 'دمياط، مصر': 4}
Payment terms (days): {15: 8, 30: 10, 45: 10, 60: 8, 90: 12}


In [46]:
# what does each kind of supplier actually sell? (purchase orders -> SKUs -> categories)
po = tables["purchase_orders"]
lines = tables["purchase_order_items"].merge(po[["po_id", "po_date", "supplier_id", "po_status"]], on="po_id")
lines = lines.merge(v[["sku_id", "product_id"]], on="sku_id").merge(p[["product_id", "category_en"]], on="product_id")
lines["material_type"] = lines.supplier_id.map(sup.set_index("supplier_id").material_type)
per_sup = lines.groupby("supplier_id").agg(skus=("sku_id", "nunique"), categories=("category_en", "nunique"))
print("SKUs bought per supplier: min", per_sup.skus.min(), "| max", per_sup.skus.max(), "(of 695) | categories:", per_sup.categories.unique())
lines.groupby("material_type").quantity.sum().div(1e6).round(1).to_frame("units bought (M)")

SKUs bought per supplier: min 650 | max 677 (of 695) | categories: [16]


,units bought (M)
material_type,
Accessories,35.9
Cotton yarn,27.8
Dyes,34.7
Elastic & Lycra,35.3
Finished garments,35.8
Knitted fabric,35.4
Labels,35.9
Logistics,28.5
Machinery services,35.0


In [47]:
# example: SUP008 is a "Dyes" supplier
lines[lines.supplier_id == "SUP008"].groupby("category_en").quantity.sum().sort_values(ascending=False).head(5).to_frame("units from SUP008 (Dyes)")

,units from SUP008 (Dyes)
category_en,
Socks,3556638
Men tops,522310
Kids underwear,476502
Women tops,364638
Men boxers,341855


In [48]:
# is any supplier cheaper, faster or more reliable than the others?
lines["year"] = lines.po_date.str[:4]
lines["cost_vs_same_sku"] = lines.unit_cost_EGP / lines.groupby(["sku_id", "year"]).unit_cost_EGP.transform("mean")
lines["lead_days"] = (pd.to_datetime(lines.delivery_date) - pd.to_datetime(lines.po_date)).dt.days
card = lines.groupby("supplier_id").agg(units=("quantity", "sum"), cost_vs_others=("cost_vs_same_sku", "mean"),
                                        lead_days=("lead_days", "mean"))
card["partial_receipt_%"] = po.groupby("supplier_id").po_status.apply(lambda s: (s == "Partial receipt").mean() * 100)
print("Cost of the same SKU, same year, from order to order (5% - 95%):", lines.cost_vs_same_sku.quantile([.05, .95]).round(2).tolist())
print("Lead days by supplier city:", lines.groupby(lines.supplier_id.map(sup.set_index("supplier_id").city_country)).lead_days.mean().round(1).to_dict(), "\n")
card.describe().loc[["min", "max"]].round(3)

Cost of the same SKU, same year, from order to order (5% - 95%): [0.74, 1.26]
Lead days by supplier city: {'الإسكندرية، مصر': 29.0, 'الجيزة، مصر': 28.8, 'القاهرة، مصر': 28.6, 'المحلة الكبرى، مصر': 29.0, 'بورسعيد، مصر': 28.9, 'دمياط، مصر': 28.7} 



,units,cost_vs_others,lead_days,partial_receipt_%
min,6499381.0,0.993,28.297,28.884
max,7676390.0,1.008,29.443,35.473


In [49]:
# payment terms: does the PO use the terms agreed with the supplier?
pt = po.merge(sup[["supplier_id", "payment_terms_days"]], on="supplier_id", suffixes=("_po", "_supplier"))
print("POs that use the supplier's terms:", f"{(pt.payment_terms_days_po == pt.payment_terms_days_supplier).mean():.1%}")
print("POs of 15-day suppliers that use 15 days:", ((pt.payment_terms_days_supplier == 15) & (pt.payment_terms_days_po == 15)).sum(),
      "of", (pt.payment_terms_days_supplier == 15).sum())
pd.crosstab(pt.payment_terms_days_supplier, pt.payment_terms_days_po, normalize="index").round(2)

POs that use the supplier's terms: 20.9%
POs of 15-day suppliers that use 15 days: 0 of 6989


payment_terms_days_po,30,45,60,90
payment_terms_days_supplier,,,,
15,0.25,0.25,0.25,0.25
30,0.25,0.24,0.25,0.25
45,0.25,0.25,0.24,0.25
60,0.25,0.25,0.26,0.25
90,0.25,0.25,0.25,0.25


In [50]:
# where does supplier_id appear?
for name, df in tables.items():
    if "supplier_id" in df.columns and name != "suppliers":
        print(f"{name:20s}", df.supplier_id.nunique(), "suppliers used |", (~df.supplier_id.isin(sup.supplier_id)).sum(), "unknown codes")

purchase_orders      48 suppliers used | 0 unknown codes


## Step 7: `factories`

In [51]:
f = tables["factories"]
print(f.shape)
f

(6, 4)


,factory_id,factory_name_ar,governorate_ar,production_focus
0,FAC01,وحدة تصنيع 1,الشرقية,Knitting / Cut & Sew / Packaging
1,FAC02,وحدة تصنيع 2,القليوبية,Knitting / Cut & Sew / Packaging
2,FAC03,وحدة تصنيع 3,القاهرة,Knitting / Cut & Sew / Packaging
3,FAC04,وحدة تصنيع 4,القليوبية,Knitting / Cut & Sew / Packaging
4,FAC05,وحدة تصنيع 5,الغربية,Knitting / Cut & Sew / Packaging
5,FAC06,وحدة تصنيع 6,القاهرة,Knitting / Cut & Sew / Packaging


In [52]:
print("Empty cells:", f.isna().sum().sum(), "| duplicate factory_id:", f.factory_id.duplicated().sum())
print("Columns in dictionary but NOT in file:", set(dd[dd.table_name == "factories.csv"].field_name) - set(f.columns))
print("Production focus values:", f.production_focus.unique().tolist())

# is a factory code used in ANY other table?
used = {name: [col for col in df.columns if pd.api.types.is_string_dtype(df[col]) and df[col].isin(f.factory_id).any()]
        for name, df in tables.items() if name != "factories"}
print("Tables that use a factory code:", {k: v for k, v in used.items() if v} or "none")
print("Where employees work:", tables["employees"].branch_id.str[:2].unique().tolist(),
      "| who has expenses:", tables["operating_expenses"].branch_or_department_id.str[:2].unique().tolist())

Empty cells: 0 | duplicate factory_id: 0
Columns in dictionary but NOT in file: set()
Production focus values: ['Knitting / Cut & Sew / Packaging']
Tables that use a factory code: none
Where employees work: ['BR'] | who has expenses: ['WH', 'BR']


## Step 8: `import_partners`

In [53]:
ipt = tables["import_partners"]
print(ipt.shape)
ipt

(16, 5)


,import_partner_id,partner_name,country,goods_category,shipping_terms
0,IMP001,Import Partner 001,Türkiye,Fabric & machinery,CIF
1,IMP002,Import Partner 002,China,Fabric & machinery,FOB
2,IMP003,Import Partner 003,Pakistan,Accessories / packaging,FOB
3,IMP004,Import Partner 004,China,Fabric & machinery,CIF
4,IMP005,Import Partner 005,Türkiye,Fabric & machinery,CIF
5,IMP006,Import Partner 006,Vietnam,Accessories / packaging,CIF
6,IMP007,Import Partner 007,Türkiye,Fabric & machinery,FOB
7,IMP008,Import Partner 008,India,Fabric & machinery,EXW
8,IMP009,Import Partner 009,Pakistan,Accessories / packaging,CIF
9,IMP010,Import Partner 010,Vietnam,Fabric & machinery,EXW


In [54]:
print("Empty cells:", ipt.isna().sum().sum(), "| duplicate id:", ipt.import_partner_id.duplicated().sum())
print("Columns in dictionary but NOT in file:", set(dd[dd.table_name == "import_partners.csv"].field_name) - set(ipt.columns))
print("Countries:", ipt.country.value_counts().to_dict())
print("Goods:", ipt.goods_category.value_counts().to_dict(), "| Shipping terms:", ipt.shipping_terms.value_counts().to_dict())

Empty cells: 0 | duplicate id: 0
Columns in dictionary but NOT in file: set()
Countries: {'China': 4, 'Pakistan': 4, 'Türkiye': 3, 'Vietnam': 2, 'India': 2, 'Italy': 1}
Goods: {'Fabric & machinery': 11, 'Accessories / packaging': 5} | Shipping terms: {'CIF': 7, 'EXW': 5, 'FOB': 4}


In [55]:
# does each partner ship what it is supposed to? (import_shipments)
ims = tables["import_shipments"].merge(ipt, on="import_partner_id")
print("Shipments per partner: min", ims.import_partner_id.value_counts().min(), "| max", ims.import_partner_id.value_counts().max())
pd.crosstab(ims.goods_category, ims.commodity_class)

Shipments per partner: min 231 | max 297


commodity_class,Accessories,Cotton fabric,Industrial machinery,Packaging materials
goods_category,,,,
Accessories / packaging,344,315,318,346
Fabric & machinery,688,757,704,728


In [56]:
# do shipping terms or country change the freight we pay?
print("Avg freight (USD) by shipping terms:", ims.groupby("shipping_terms").freight_USD.mean().round(0).to_dict())
print("Avg freight (USD) by country:", ims.groupby("country").freight_USD.mean().round(0).to_dict(), "\n")
pd.crosstab(ims.country, ims.port_name, normalize="index").round(2)

Avg freight (USD) by shipping terms: {'CIF': 6162.0, 'EXW': 6093.0, 'FOB': 6195.0}
Avg freight (USD) by country: {'China': 6164.0, 'India': 6241.0, 'Italy': 6294.0, 'Pakistan': 6053.0, 'Türkiye': 6154.0, 'Vietnam': 6145.0} 



port_name,Alexandria Port,Port Said Port,Sokhna Port
country,,,
China,0.33,0.34,0.33
India,0.32,0.32,0.36
Italy,0.34,0.28,0.38
Pakistan,0.32,0.36,0.32
Türkiye,0.34,0.33,0.33
Vietnam,0.30,0.31,0.39


## Step 9: `employees`

In [57]:
e = tables["employees"]
print(e.shape)
e.head(12)

(640, 6)


,employee_id,employee_name_ar,branch_id,position_en,hire_date,salary_base_EGP_2026
0,EMP00001,منى فتحى,BR001,Cashier,10/19/2022,17300
1,EMP00002,يوسف عبدالعزيز,BR002,Sales Associate,9/19/2019,14900
2,EMP00003,منى حسن,BR003,Sales Associate,1/19/2020,12200
3,EMP00004,عمر حسن,BR004,Store Manager,4/19/2023,17200
4,EMP00005,رانيا محمود,BR005,Inventory Clerk,11/19/2023,16100
5,EMP00006,أسماء عبدالفتاح,BR006,Area Manager,11/19/2023,13800
6,EMP00007,طارق الدسوقي,BR007,Cashier,2/19/2021,10400
7,EMP00008,أمل حسن,BR008,Sales Associate,3/19/2021,13800
8,EMP00009,دعاء الشريف,BR009,Sales Associate,10/19/2019,16900
9,EMP00010,عمر محمد,BR010,Store Manager,3/19/2021,7400


In [58]:
print("Empty cells:", e.isna().sum().sum(), "| duplicate employee_id:", e.employee_id.duplicated().sum())
print("Columns in dictionary but NOT in file:", set(dd[dd.table_name == "employees.csv"].field_name) - set(e.columns))
print("Employees per branch:", e.groupby("branch_id").size().value_counts().to_dict(), "| unknown branch codes:", (~e.branch_id.isin(b.branch_id)).sum())
print("Positions:", e.position_en.value_counts().to_dict())
e["hire_date"] = pd.to_datetime(e.hire_date, format="%m/%d/%Y")
print("Hire day of month:", e.hire_date.dt.day.value_counts().to_dict(), "| range:", e.hire_date.min().date(), "->", e.hire_date.max().date())
print("Hires per year:", e.hire_date.dt.year.value_counts().sort_index().to_dict())

Empty cells: 0 | duplicate employee_id: 0
Columns in dictionary but NOT in file: set()
Employees per branch: {8: 80} | unknown branch codes: 0
Positions: {'Sales Associate': 214, 'Cashier': 107, 'Store Manager': 107, 'Inventory Clerk': 106, 'Area Manager': 106}
Hire day of month: {19: 640} | range: 2018-10-19 -> 2024-03-19
Hires per year: {2018: 22, 2019: 104, 2020: 138, 2021: 116, 2022: 126, 2023: 106, 2024: 28}


In [59]:
# roles go in a fixed circle: employee 1 Cashier, 2 Sales, 3 Sales, 4 Store Manager, 5 Inventory Clerk, 6 Area Manager, 7 Cashier ...
roles = pd.crosstab(e.branch_id, e.position_en)
print("Branches with NO store manager:", (roles["Store Manager"] == 0).sum(), "| NO cashier:", (roles["Cashier"] == 0).sum(),
      "| both:", ((roles["Store Manager"] > 0) & (roles["Cashier"] > 0)).sum())
print("Area managers inside one branch:", roles["Area Manager"].value_counts().sort_index().to_dict())
roles.head(4)

Branches with NO store manager: 40 | NO cashier: 40 | both: 0
Area managers inside one branch: {0: 40, 2: 14, 3: 26}


position_en,Area Manager,Cashier,Inventory Clerk,Sales Associate,Store Manager
branch_id,,,,,
BR001,0,3,2,3,0
BR002,2,0,0,3,3
BR003,0,2,3,3,0
BR004,3,0,0,2,3


In [60]:
# salary vs position and years of service
tenure_years = (pd.Timestamp("2026-10-18") - e.hire_date).dt.days / 365.25
print("Correlation salary ~ years of service:", round(e.salary_base_EGP_2026.corr(tenure_years), 3))
e.groupby("position_en").salary_base_EGP_2026.agg(["count", "mean", "min", "max"]).round(0).sort_values("mean")

Correlation salary ~ years of service: -0.013


,count,mean,min,max
position_en,,,,
Inventory Clerk,106,11539.0,5200,18000
Area Manager,106,11691.0,5300,18400
Sales Associate,214,11899.0,5300,18300
Cashier,107,11959.0,5300,18400
Store Manager,107,12487.0,5400,18400


In [61]:
# where does employee_id appear? (is the seller recorded on any order?)
for name, df in tables.items():
    cols = [col for col in df.columns if "employee" in col]
    if cols and name != "employees":
        print(f"{name:20s}", cols)

payroll              ['employee_id']


## Step 10: `sales_orders`

In [62]:
so = tables["sales_orders"]
print(so.shape)
so.head()

(540000, 13)


,sales_order_id,order_date,branch_id,customer_id,sales_channel,payment_method,items_count,gross_EGP,discount_EGP,net_before_tax_EGP,tax_EGP,paid_total_EGP,status
0,SO00000001,2018-10-19,BR010,C0000006,Retail POS,Card POS,1,27.00,2.70,24.30,3.40,27.70,Completed
1,SO00000002,2018-10-19,BR047,C0000010,Retail POS,Cash,2,279.35,10.80,268.55,37.60,306.15,Completed
2,SO00000003,2018-10-19,BR013,NaN,E-commerce,Card POS,1,160.72,32.14,128.58,18.00,146.58,Delivered
3,SO00000004,2018-10-19,BR029,C0000025,Retail POS,Bank transfer,1,32.06,1.60,30.46,4.26,34.72,Completed
4,SO00000005,2018-10-19,BR009,NaN,Retail POS,Bank transfer,3,293.69,3.65,290.04,40.61,330.65,Completed


In [63]:
print("Empty cells:", so.isna().sum()[lambda s: s > 0].to_dict(), "| duplicate id:", so.sales_order_id.duplicated().sum())
print("Columns in dictionary but NOT in file:", set(dd[dd.table_name == "sales_orders.csv"].field_name) - set(so.columns))
print("Date range:", so.order_date.min(), "->", so.order_date.max())
for col in ["sales_channel", "payment_method", "status"]:
    print(col, so[col].value_counts().to_dict())
print("Totals (M EGP): paid", round(so.paid_total_EGP.sum() / 1e6, 1), "| net", round(so.net_before_tax_EGP.sum() / 1e6, 1),
      "| tax", round(so.tax_EGP.sum() / 1e6, 1), "| discount", round(so.discount_EGP.sum() / 1e6, 1))

Empty cells: {'customer_id': 102961} | duplicate id: 0
Columns in dictionary but NOT in file: set()
Date range: 2018-10-19 -> 2026-10-18
sales_channel {'Retail POS': 388171, 'E-commerce': 97660, 'Wholesale / reseller': 54169}
payment_method {'Cash': 215552, 'Card POS': 183239, 'Wallet': 59714, 'Cash on delivery': 43476, 'Bank transfer': 38019}
status {'Completed': 448120, 'Delivered': 81152, 'Partially returned': 10728}
Totals (M EGP): paid 959.1 | net 841.3 | tax 117.8 | discount 37.9


In [64]:
# math inside each row, and each order vs its lines
print("gross - discount = net:", f"{(abs(so.gross_EGP - so.discount_EGP - so.net_before_tax_EGP) < 0.011).mean():.0%}")
print("tax / net:", (so.tax_EGP / so.net_before_tax_EGP).round(2).unique())
print("net + tax = paid:", f"{(abs(so.net_before_tax_EGP + so.tax_EGP - so.paid_total_EGP) < 0.011).mean():.0%}")
lines_sum = si.groupby("sales_order_id").agg(lines=("sales_line_id", "size"), units=("quantity", "sum"), net=("net_line_EGP", "sum"))
chk = so.set_index("sales_order_id").join(lines_sum)
print("order net = sum of its lines:", f"{(abs(chk.net_before_tax_EGP - chk.net) < 0.05).mean():.0%}",
      "| items_count = lines:", f"{(chk.items_count == chk.lines).mean():.0%}", "| items_count = units:", f"{(chk.items_count == chk.units).mean():.0%}")

gross - discount = net: 100%
tax / net: [0.14]
net + tax = paid: 100%
order net = sum of its lines: 100% | items_count = lines: 100% | items_count = units: 24%


In [65]:
# real patterns: growth and inflation (data year = Oct 19 -> Oct 18, because 2018 and 2026 are not full)
so["date"] = pd.to_datetime(so.order_date)
so["data_year"] = so.date.dt.year - (so.date.dt.strftime("%m-%d") < "10-19")
yearly = so.groupby("data_year").agg(orders=("sales_order_id", "size"), avg_paid=("paid_total_EGP", "mean"), paid_M=("paid_total_EGP", "sum"))
yearly["paid_M"] = yearly.paid_M / 1e6
yearly["orders_growth_%"] = yearly.orders.pct_change() * 100
yearly.round(1)

,orders,avg_paid,paid_M,orders_growth_%
data_year,,,,
2018,42314,543.5,23.0,NaN
2019,45646,618.0,28.2,7.9
2020,51836,685.0,35.5,13.6
2021,59835,900.1,53.9,15.4
2022,69692,1239.2,86.4,16.5
2023,79749,1913.8,152.6,14.4
2024,90650,2705.7,245.3,13.7
2025,100278,3333.0,334.2,10.6


In [66]:
# real patterns: season, weekday, basket size
monthly = so.groupby(so.date.dt.to_period("M")).size()
season = (monthly / monthly.rolling(12, center=True).mean()).dropna()
print("Season index by month (1 = normal):", season.groupby(season.index.month).mean().round(2).to_dict())
print("Share by weekday %:", (so.date.dt.day_name().value_counts(normalize=True) * 100).round(1).to_dict())
units = si.groupby("sales_order_id").quantity.sum()
print("Units per order by channel:", so.sales_order_id.map(units).groupby(so.sales_channel).mean().round(1).to_dict())

Season index by month (1 = normal): {1: 0.98, 2: 0.9, 3: 1.11, 4: 1.08, 5: 1.1, 6: 0.91, 7: 0.93, 8: 0.92, 9: 0.89, 10: 0.9, 11: 1.17, 12: 1.18}
Share by weekday %: {'Thursday': 15.3, 'Friday': 15.3, 'Saturday': 15.3, 'Wednesday': 13.6, 'Sunday': 13.6, 'Tuesday': 13.5, 'Monday': 13.5}
Units per order by channel: {'E-commerce': 4.4, 'Retail POS': 4.3, 'Wholesale / reseller': 23.9}


In [67]:
# channel share every year (online stays ~18%, even in 2020)
print("Spring orders 2019 vs 2020 (lockdown):",
      {m: (int(((so.date.dt.year == 2019) & (so.date.dt.month == m)).sum()), int(((so.date.dt.year == 2020) & (so.date.dt.month == m)).sum())) for m in [3, 4, 5]})
pd.crosstab(so.data_year, so.sales_channel, normalize="index").round(3)

Spring orders 2019 vs 2020 (lockdown): {3: (3895, 4137), 4: (3787, 4124), 5: (4073, 4080)}


sales_channel,E-commerce,Retail POS,Wholesale / reseller
data_year,,,
2018,0.179,0.720,0.101
2019,0.176,0.721,0.103
2020,0.182,0.717,0.101
2021,0.180,0.719,0.101
2022,0.182,0.718,0.101
2023,0.179,0.720,0.102
2024,0.184,0.716,0.100
2025,0.183,0.721,0.097


In [68]:
# payment method and status vs channel
print(pd.crosstab(so.sales_channel, so.payment_method), "\n")
shipped = so.sales_order_id.isin(tables["delivery_shipments"].sales_order_id).rename("has_shipment")
pd.crosstab([so.sales_channel, shipped], so.status)

payment_method        Bank transfer  Card POS    Cash  Cash on delivery  Wallet
sales_channel                                                                  
E-commerce                     6842     33113   39173              7773   10759
Retail POS                    27395    131632  154799             31333   43012
Wholesale / reseller           3782     18494   21580              4370    5943 



,status,Completed,Delivered,Partially returned
sales_channel,has_shipment,,,
E-commerce,True,81162,14588,1910
Retail POS,False,322020,58374,7777
Wholesale / reseller,False,44938,8190,1041


In [69]:
# status "Partially returned" vs the returns table
has_return = so.sales_order_id.isin(tables["sales_returns"].sales_order_id).rename("has_return_record")
pd.crosstab(so.status, has_return)

has_return_record,False,True
status,,
Completed,424432,23688
Delivered,76900,4252
Partially returned,10168,560


## Step 11: `sales_order_items`

In [70]:
print(si.shape)
si.head()

(1138536, 13)


,sales_line_id,sales_order_id,order_date,branch_id,sku_id,product_id,quantity,unit_price_EGP,gross_line_EGP,discount_line_EGP,net_line_EGP,unit_cost_EGP,margin_EGP
0,SL000000001,SO00000001,2018-10-19,BR010,SKU000662,PR0133,1,27.00,27.00,2.70,24.30,12.95,11.35
1,SL000000002,SO00000002,2018-10-19,BR047,SKU000532,PR0107,1,71.99,71.99,10.80,61.19,48.74,12.45
2,SL000000003,SO00000002,2018-10-19,BR047,SKU000374,PR0075,6,34.56,207.36,0.00,207.36,16.68,107.28
3,SL000000004,SO00000003,2018-10-19,BR013,SKU000602,PR0121,1,160.72,160.72,32.14,128.58,85.44,43.14
4,SL000000005,SO00000004,2018-10-19,BR029,SKU000370,PR0074,1,32.06,32.06,1.60,30.46,21.09,9.37


In [71]:
print("Empty cells:", si.isna().sum().sum(), "| duplicate id:", si.sales_line_id.duplicated().sum())
print("Columns in dictionary but NOT in file:", set(dd[dd.table_name == "sales_order_items.csv"].field_name) - set(si.columns))
print("qty x price = gross:", f"{(abs(si.quantity * si.unit_price_EGP - si.gross_line_EGP) < 0.011).mean():.0%}",
      "| gross - discount = net:", f"{(abs(si.gross_line_EGP - si.discount_line_EGP - si.net_line_EGP) < 0.011).mean():.0%}",
      "| net - qty x cost = margin:", f"{(abs(si.net_line_EGP - si.quantity * si.unit_cost_EGP - si.margin_EGP) < 0.011).mean():.0%}")
same = si[["sales_order_id", "order_date", "branch_id"]].merge(so[["sales_order_id", "order_date", "branch_id"]], on="sales_order_id")
print("order_date & branch_id same as the order:", f"{((same.order_date_x == same.order_date_y) & (same.branch_id_x == same.branch_id_y)).mean():.0%}")
print("Totals: units", si.quantity.sum(), "| net (M)", round(si.net_line_EGP.sum() / 1e6, 1), "| margin (M)", round(si.margin_EGP.sum() / 1e6, 1),
      "| margin %", round(si.margin_EGP.sum() / si.net_line_EGP.sum() * 100, 1))

Empty cells: 0 | duplicate id: 0
Columns in dictionary but NOT in file: set()
qty x price = gross: 100% | gross - discount = net: 100% | net - qty x cost = margin: 100%
order_date & branch_id same as the order: 100%
Totals: units 3398655 | net (M) 841.3 | margin (M) 322.6 | margin % 38.3


In [72]:
# real pattern: prices went up ~7x (selling price vs the 2026 website price of the same SKU)
sl = si.merge(p[["product_id", "listed_price_EGP", "category_en"]], on="product_id").merge(v[["sku_id", "price_multiplier"]], on="sku_id")
sl["year"] = sl.order_date.str[:4]
price_index = (sl.unit_price_EGP / (sl.listed_price_EGP * sl.price_multiplier)).groupby(sl.year).mean()
pd.DataFrame({"price_vs_2026": price_index.round(3), "change_%": (price_index.pct_change() * 100).round(0)})

,price_vs_2026,change_%
year,,
2018,0.141,NaN
2019,0.162,14.0
2020,0.182,13.0
2021,0.202,11.0
2022,0.273,35.0
2023,0.384,41.0
2024,0.596,55.0
2025,0.828,39.0
2026,1.010,22.0


In [73]:
# real pattern: category mix (revenue vs pieces)
mix = sl.groupby("category_en").agg(units=("quantity", "sum"), net=("net_line_EGP", "sum"))
mix["units_%"] = (mix.units / mix.units.sum() * 100).round(1)
mix["revenue_%"] = (mix.net / mix.net.sum() * 100).round(1)
mix.sort_values("net", ascending=False)[["units_%", "revenue_%"]].head(6)

,units_%,revenue_%
category_en,,
Men boxers,11.2,23.1
Men tops,13.0,19.6
Socks,26.8,13.8
Men underwear,5.6,6.3
Women pajamas,2.3,6.0
Women tops,10.0,5.8


In [74]:
# discount: who gets it? (the same ~54% every month, channel and category)
sl["discounted"] = sl.discount_line_EGP > 0
print("Discounted lines by month:", sl.groupby(sl.order_date.str[5:7]).discounted.mean().round(2).to_dict())
by_cat_disc = sl.groupby("category_en").discounted.mean()
print("Discounted lines by category: min", round(by_cat_disc.min(), 3), "| max", round(by_cat_disc.max(), 3))
print("Discount sizes (share of gross):", (sl.discount_line_EGP / sl.gross_line_EGP)[sl.discounted].round(2).value_counts().sort_index().to_dict())

Discounted lines by month: {'01': 0.54, '02': 0.54, '03': 0.54, '04': 0.54, '05': 0.54, '06': 0.54, '07': 0.54, '08': 0.54, '09': 0.54, '10': 0.54, '11': 0.54, '12': 0.54}
Discounted lines by category: min 0.534 | max 0.548
Discount sizes (share of gross): {0.03: 136608, 0.05: 193724, 0.1: 170449, 0.15: 91092, 0.2: 22698}


In [75]:
# does a discount sell more pieces? and what does it cost?
depth = (sl.discount_line_EGP / sl.gross_line_EGP).round(2)
print("Avg pieces per line by discount size:", sl.quantity.groupby(depth).mean().round(2).to_dict())
print("Margin % without / with discount:", (sl.groupby("discounted").margin_EGP.sum() / sl.groupby("discounted").net_line_EGP.sum() * 100).round(1).to_dict())
print("Total discount (M EGP):", round(sl.discount_line_EGP.sum() / 1e6, 1))

Avg pieces per line by discount size: {0.0: 2.98, 0.03: 2.98, 0.05: 3.0, 0.1: 2.98, 0.15: 2.99, 0.2: 2.99}
Margin % without / with discount: {False: 41.0, True: 35.9}
Total discount (M EGP): 37.9


In [76]:
# cost is always ~59% of the price -> the same margin everywhere
print("cost / price by year:", (sl.unit_cost_EGP / sl.unit_price_EGP).groupby(sl.year).mean().round(2).to_dict())
by_cat = sl.groupby("category_en").margin_EGP.sum() / sl.groupby("category_en").net_line_EGP.sum() * 100
by_prod = sl.groupby("product_id").margin_EGP.sum() / sl.groupby("product_id").net_line_EGP.sum() * 100
print("Margin % by category: min", round(by_cat.min(), 1), "| max", round(by_cat.max(), 1),
      "|| by product: min", round(by_prod.min(), 1), "| max", round(by_prod.max(), 1))

cost / price by year: {'2018': 0.59, '2019': 0.59, '2020': 0.59, '2021': 0.59, '2022': 0.59, '2023': 0.59, '2024': 0.59, '2025': 0.59, '2026': 0.59}
Margin % by category: min 38.0 | max 38.5 || by product: min 37.6 | max 39.4


## Step 12: `sales_payments`

In [77]:
sp = tables["sales_payments"]
print(sp.shape)
sp.head()

(540000, 6)


,payment_id,sales_order_id,payment_date,payment_method,amount_EGP,status
0,PAY00000001,SO00000001,2018-10-19,Card POS,27.70,Paid
1,PAY00000002,SO00000002,2018-10-19,Cash,306.15,Paid
2,PAY00000003,SO00000003,2018-10-19,Card POS,146.58,Paid
3,PAY00000004,SO00000004,2018-10-19,Bank transfer,34.72,Paid
4,PAY00000005,SO00000005,2018-10-19,Bank transfer,330.65,Paid


In [78]:
print("Empty cells:", sp.isna().sum().sum(), "| duplicate id:", sp.payment_id.duplicated().sum())
print("Columns in dictionary but NOT in file:", set(dd[dd.table_name == "sales_payments.csv"].field_name) - set(sp.columns))
print("Status:", sp.status.value_counts().to_dict())
print("Payments per order:", sp.groupby("sales_order_id").size().value_counts().to_dict(),
      "| orders with no payment:", (~so.sales_order_id.isin(sp.sales_order_id)).sum(), "| total (M EGP):", round(sp.amount_EGP.sum() / 1e6, 1))

Empty cells: 0 | duplicate id: 0
Columns in dictionary but NOT in file: set()
Status: {'Paid': 540000}
Payments per order: {1: 540000} | orders with no payment: 0 | total (M EGP): 959.1


In [79]:
# is it just a copy of sales_orders?
pay = sp.merge(so[["sales_order_id", "order_date", "payment_method", "paid_total_EGP", "sales_channel"]], on="sales_order_id", suffixes=("", "_order"))
print("amount = order paid_total:", f"{(abs(pay.amount_EGP - pay.paid_total_EGP) < 0.011).mean():.0%}",
      "| same method:", f"{(pay.payment_method == pay.payment_method_order).mean():.0%}",
      "| paid on the order day:", f"{(pay.payment_date == pay.order_date).mean():.0%}")

amount = order paid_total: 100% | same method: 100% | paid on the order day: 100%


In [80]:
# cash on delivery: paid before it was delivered? and where are the shipping fees?
cod = pay[(pay.payment_method == "Cash on delivery") & (pay.sales_channel == "E-commerce")].merge(
    tables["delivery_shipments"][["sales_order_id", "delivery_date"]], on="sales_order_id")
days_before = (pd.to_datetime(cod.delivery_date) - pd.to_datetime(cod.payment_date)).dt.days
print("Online COD orders:", len(cod), "| paid BEFORE delivery:", (days_before > 0).sum(), "| avg days before:", round(days_before.mean(), 1))
print("Shipping fees in delivery_shipments (M EGP):", round(tables["delivery_shipments"].shipping_fee_EGP.sum() / 1e6, 2), "-> not inside any payment amount")

Online COD orders: 7773 | paid BEFORE delivery: 7769 | avg days before: 5.0
Shipping fees in delivery_shipments (M EGP): 5.01 -> not inside any payment amount


## Step 13: `sales_returns`

In [81]:
sr = tables["sales_returns"]
print(sr.shape)
sr.head()

(28500, 9)


,return_id,sales_order_id,sales_line_id,return_date,sku_id,return_quantity,refund_amount_EGP,reason,refund_method
0,RET0000001,SO00472785,SL000996848,2026-02-20,SKU000676,1,637.69,Size issue,Cash
1,RET0000002,SO00228394,SL000481462,2023-04-01,SKU000599,1,290.60,Defect claim,Cash
2,RET0000003,SO00307618,SL000648623,2024-04-15,SKU000533,3,931.99,Wrong item,Wallet
3,RET0000004,SO00374944,SL000790483,2025-02-12,SKU000278,1,215.40,Exchange,Wallet
4,RET0000005,SO00275488,SL000580915,2023-11-26,SKU000176,10,1040.96,Other,Wallet


In [82]:
print("Empty cells:", sr.isna().sum().sum(), "| duplicate id:", sr.return_id.duplicated().sum())
print("Columns in dictionary but NOT in file:", set(dd[dd.table_name == "sales_returns.csv"].field_name) - set(sr.columns))
print("Reasons:", sr.reason.value_counts().to_dict())
print("Refund methods:", sr.refund_method.value_counts().to_dict())
print("Returned units:", sr.return_quantity.sum(), "of", si.quantity.sum(), f"sold ({sr.return_quantity.sum() / si.quantity.sum():.1%})",
      "| refunds (M EGP):", round(sr.refund_amount_EGP.sum() / 1e6, 2))

Empty cells: 0 | duplicate id: 0
Columns in dictionary but NOT in file: set()
Reasons: {'Other': 5773, 'Size issue': 5750, 'Defect claim': 5691, 'Exchange': 5644, 'Wrong item': 5642}
Refund methods: {'Card reversal': 9650, 'Cash': 9441, 'Wallet': 9409}
Returned units: 57136 of 3398655 sold (1.7%) | refunds (M EGP): 13.91


In [83]:
# link to the sold line: same order and SKU? do quantity, dates and refund make sense?
r = sr.merge(si[["sales_line_id", "sales_order_id", "sku_id", "order_date", "quantity", "net_line_EGP"]], on="sales_line_id", suffixes=("", "_line"))
print("same order & SKU as the line:", f"{((r.sales_order_id == r.sales_order_id_line) & (r.sku_id == r.sku_id_line)).mean():.0%}",
      "| returned more than bought:", (r.return_quantity > r.quantity).sum(), "| full line returned:", f"{(r.return_quantity == r.quantity).mean():.0%}")
r["days_after_order"] = (pd.to_datetime(r.return_date) - pd.to_datetime(r.order_date)).dt.days
print("Days from order to return:", r.days_after_order.min(), "->", r.days_after_order.max(), "| avg", round(r.days_after_order.mean(), 1))
returned_net = r.net_line_EGP * r.return_quantity / r.quantity
print("Refund / net price of the returned pieces (min, max):", (r.refund_amount_EGP / returned_net).agg(["min", "max"]).round(2).tolist())
print("Refunds MORE than the customer paid incl. 14% VAT:", (r.refund_amount_EGP > returned_net * 1.14 + 0.01).sum())

same order & SKU as the line: 100% | returned more than bought: 0 | full line returned: 64%
Days from order to return: 0 -> 24 | avg 12.4
Refund / net price of the returned pieces (min, max): [0.87, 1.25]
Refunds MORE than the customer paid incl. 14% VAT: 922


In [84]:
# reasons: "Size issue" even for Free-size items, and "Exchange" but money refunded
r = r.merge(v[["sku_id", "size"]], on="sku_id")
print(pd.crosstab((r["size"] == "Free").rename("free_size"), r.reason, normalize="index").round(2), "\n")
print("Exchange returns:", (r.reason == "Exchange").sum(), "| money refunded on them (M EGP):", round(r[r.reason == "Exchange"].refund_amount_EGP.sum() / 1e6, 2))

reason     Defect claim  Exchange  Other  Size issue  Wrong item
free_size                                                       
False              0.20       0.2    0.2        0.20        0.20
True               0.21       0.2    0.2        0.19        0.21 

Exchange returns: 5644 | money refunded on them (M EGP): 2.74


In [85]:
# refund method vs how the customer paid
r = r.merge(so[["sales_order_id", "payment_method", "sales_channel"]], on="sales_order_id")
pd.crosstab(r.payment_method, r.refund_method, normalize="index").round(2)

refund_method,Card reversal,Cash,Wallet
payment_method,,,
Bank transfer,0.33,0.34,0.33
Card POS,0.35,0.33,0.33
Cash,0.34,0.33,0.33
Cash on delivery,0.32,0.34,0.34
Wallet,0.34,0.32,0.33


In [86]:
# return rate by channel and category, and online returns dated before the delivery
li = si[["sales_line_id", "sales_order_id", "quantity", "product_id"]].merge(so[["sales_order_id", "sales_channel"]], on="sales_order_id")
li = li.merge(p[["product_id", "category_en"]], on="product_id")
li["returned"] = li.sales_line_id.map(sr.groupby("sales_line_id").return_quantity.sum()).fillna(0)
by_ch = li.groupby("sales_channel")[["returned", "quantity"]].sum()
by_cat_r = li.groupby("category_en")[["returned", "quantity"]].sum()
print("Return rate % by channel:", (by_ch.returned / by_ch.quantity * 100).round(2).to_dict())
print("Return rate % by category: min", round((by_cat_r.returned / by_cat_r.quantity * 100).min(), 2), "| max", round((by_cat_r.returned / by_cat_r.quantity * 100).max(), 2))
online = r[r.sales_channel == "E-commerce"].merge(tables["delivery_shipments"][["sales_order_id", "delivery_date"]], on="sales_order_id")
print("Online returns dated BEFORE the delivery:", (pd.to_datetime(online.return_date) < pd.to_datetime(online.delivery_date)).sum(), "of", len(online))

Return rate % by channel: {'E-commerce': 1.84, 'Retail POS': 1.85, 'Wholesale / reseller': 1.41}
Return rate % by category: min 1.52 | max 1.78
Online returns dated BEFORE the delivery: 888 of 5140


## Step 14: `delivery_shipments`

In [87]:
ds = tables["delivery_shipments"]
print(ds.shape)
ds.head()

(97660, 8)


,shipment_id,sales_order_id,ship_date,delivery_date,ship_to_governorate,carrier,shipping_fee_EGP,fulfillment_status
0,SHP00000001,SO00000003,2018-10-20,2018-10-23,الدقهلية,In-house delivery,60,Delivered
1,SHP00000002,SO00000008,2018-10-20,2018-10-23,أسوان,Express carrier,35,Delivered
2,SHP00000003,SO00000009,2018-10-21,2018-10-24,المنوفية,Local courier,25,Delivered
3,SHP00000004,SO00000013,2018-10-21,2018-10-26,البحر الأحمر,In-house delivery,35,Delivered
4,SHP00000005,SO00000022,2018-10-20,2018-10-22,المنيا,Local courier,25,Delivered


In [88]:
print("Empty cells:", ds.isna().sum().sum(), "| duplicate id:", ds.shipment_id.duplicated().sum(), "| orders with 2+ shipments:", ds.sales_order_id.duplicated().sum())
print("Columns in dictionary but NOT in file:", set(dd[dd.table_name == "delivery_shipments.csv"].field_name) - set(ds.columns))
print("Carriers:", ds.carrier.value_counts().to_dict(), "| Status:", ds.fulfillment_status.value_counts().to_dict())
online_orders = so[so.sales_channel == "E-commerce"].sales_order_id
print("Online orders with a shipment:", f"{online_orders.isin(ds.sales_order_id).mean():.0%}",
      "| shipments for non-online orders:", (~ds.sales_order_id.isin(online_orders)).sum())

Empty cells: 0 | duplicate id: 0 | orders with 2+ shipments: 0
Columns in dictionary but NOT in file: set()
Carriers: {'In-house delivery': 32641, 'Local courier': 32577, 'Express carrier': 32442} | Status: {'Delivered': 94705, 'Delayed': 2955}
Online orders with a shipment: 100% | shipments for non-online orders: 0


In [89]:
sh = ds.merge(so[["sales_order_id", "order_date", "customer_id", "net_before_tax_EGP"]], on="sales_order_id")
sh["days_to_ship"] = (pd.to_datetime(sh.ship_date) - pd.to_datetime(sh.order_date)).dt.days
sh["delivery_days"] = (pd.to_datetime(sh.delivery_date) - pd.to_datetime(sh.ship_date)).dt.days
print("Days from order to ship:", sh.days_to_ship.value_counts().sort_index().to_dict())
print("Delivery days:", sh.delivery_days.value_counts().sort_index().to_dict())
print("Avg delivery days - Aswan:", round(sh[sh.ship_to_governorate == "أسوان"].delivery_days.mean(), 2),
      "| Alexandria:", round(sh[sh.ship_to_governorate == "الإسكندرية"].delivery_days.mean(), 2),
      "| by carrier:", sh.groupby("carrier").delivery_days.mean().round(2).to_dict())

Days from order to ship: {0: 32690, 1: 32256, 2: 32714}
Delivery days: {0: 83, 1: 13951, 2: 14155, 3: 13809, 4: 14109, 5: 13851, 6: 13782, 7: 13920}
Avg delivery days - Aswan: 4.04 | Alexandria: 3.93 | by carrier: {'Express carrier': 3.99, 'In-house delivery': 3.99, 'Local courier': 3.98}


In [90]:
# "Delayed" vs the real number of days
print("Delayed share by delivery days:", sh.groupby("delivery_days").fulfillment_status.apply(lambda s: (s == "Delayed").mean()).round(3).to_dict())
print("Delayed but arrived in <= 1 day:", ((sh.fulfillment_status == "Delayed") & (sh.delivery_days <= 1)).sum(),
      "| 'Delivered' (on time) after 7 days:", ((sh.fulfillment_status == "Delivered") & (sh.delivery_days == 7)).sum())

Delayed share by delivery days: {0: 0.0, 1: 0.031, 2: 0.029, 3: 0.033, 4: 0.029, 5: 0.031, 6: 0.03, 7: 0.029}
Delayed but arrived in <= 1 day: 428 | 'Delivered' (on time) after 7 days: 13512


In [91]:
# shipping fee: does it depend on anything?
print("Fee values:", sh.shipping_fee_EGP.value_counts().sort_index().to_dict())
value_group = pd.qcut(sh.net_before_tax_EGP, 5, labels=["cheapest 20%", "2", "3", "4", "most expensive 20%"])
print("Free shipping share by order value:", sh.shipping_fee_EGP.eq(0).groupby(value_group, observed=True).mean().round(3).to_dict())
print("Avg fee by carrier:", sh.groupby("carrier").shipping_fee_EGP.mean().round(1).to_dict())
print("Avg fee by year:", sh.groupby(sh.order_date.str[:4]).shipping_fee_EGP.mean().round(1).to_dict(), "| total fees (M EGP):", round(sh.shipping_fee_EGP.sum() / 1e6, 2))

Fee values: {0: 14056, 25: 14265, 35: 13736, 45: 13906, 60: 13792, 85: 13867, 110: 14038}
Free shipping share by order value: {'cheapest 20%': 0.146, '2': 0.14, '3': 0.145, '4': 0.142, 'most expensive 20%': 0.146}
Avg fee by carrier: {'Express carrier': 51.1, 'In-house delivery': 51.4, 'Local courier': 51.5}
Avg fee by year: {'2018': 51.1, '2019': 51.8, '2020': 51.8, '2021': 51.3, '2022': 51.6, '2023': 51.2, '2024': 51.4, '2025': 51.1, '2026': 51.0} | total fees (M EGP): 5.01


In [92]:
# is the shipment going to the customer's governorate?
sh = sh.merge(c[["customer_id", "governorate_ar"]], on="customer_id", how="left")
known = sh.dropna(subset=["governorate_ar"])
print("Shipped to the customer's own governorate:", f"{(known.ship_to_governorate == known.governorate_ar).mean():.1%}", "(random with 21 governorates = 4.8%)")

Shipped to the customer's own governorate: 4.7% (random with 21 governorates = 4.8%)


## Step 15: `monthly_stock`

In [93]:
ms = tables["monthly_stock"]
print(ms.shape)
ms.head()

(142784, 7)


,year_month,stock_location_id,stock_location_type,product_category,closing_units,closing_stock_value_EGP,stockout_risk_score
0,2018-10,BR001,Branch,Accessories,74,2103.08,0.698
1,2018-10,BR001,Branch,Boys pajamas,66,1884.96,0.156
2,2018-10,BR001,Branch,Girls pajamas,465,18618.60,0.063
3,2018-10,BR001,Branch,Girls underwear,261,10048.50,0.540
4,2018-10,BR001,Branch,Home textiles,175,5463.50,0.077


In [94]:
print("Empty cells:", ms.isna().sum().sum(), "| duplicate (month, place, category):", ms.duplicated(["year_month", "stock_location_id", "product_category"]).sum())
print("Columns in dictionary but NOT in file:", set(dd[dd.table_name == "monthly_stock.csv"].field_name) - set(ms.columns))
print("Months:", ms.year_month.nunique(), "| places:", ms.stock_location_id.nunique(), "| categories:", ms.product_category.nunique(),
      "| rows per month:", ms.groupby("year_month").size().unique(), "| same categories as the catalog:", set(ms.product_category) == set(p.category_en))
units_by_type = ms.groupby(["year_month", "stock_location_type"]).closing_units.sum().unstack()
print("Units (branches / warehouses) first month:", units_by_type.iloc[0].to_dict(), "| last month:", units_by_type.iloc[-1].to_dict())
value = ms.groupby("year_month").closing_stock_value_EGP.sum() / 1e6
print("Stock value (M EGP): first", round(value.iloc[0], 1), "| last", round(value.iloc[-1], 1))
by_year = ms.groupby(ms.year_month.str[:4])[["closing_stock_value_EGP", "closing_units"]].sum()
print("Value per unit by year:", (by_year.closing_stock_value_EGP / by_year.closing_units).round(1).to_dict())

Empty cells: 0 | duplicate (month, place, category): 0
Columns in dictionary but NOT in file: set()
Months: 97 | places: 92 | categories: 16 | rows per month: [1472] | same categories as the catalog: True
Units (branches / warehouses) first month: {'Branch': 325457, 'Warehouse': 1528847} | last month: {'Branch': 315812, 'Warehouse': 1499027}
Stock value (M EGP): first 53.9 | last 346.0
Value per unit by year: {'2018': 27.5, '2019': 31.3, '2020': 35.8, '2021': 39.7, '2022': 53.0, '2023': 75.3, '2024': 116.8, '2025': 163.2, '2026': 195.7}


In [95]:
# branch stock vs what came in (transfers) and went out (sales), per branch, category and month
sl_cat = si.merge(p[["product_id", "category_en"]], on="product_id")
sold = sl_cat.groupby(["branch_id", "category_en", sl_cat.order_date.str[:7].rename("year_month")]).quantity.sum().rename("sold")
tr = tables["inventory_transfer_items"].merge(tables["inventory_transfers"][["transfer_id", "transfer_date", "destination_branch_id"]], on="transfer_id")
tr = tr.merge(v[["sku_id", "product_id"]], on="sku_id").merge(p[["product_id", "category_en"]], on="product_id")
received = tr.groupby([tr.destination_branch_id.rename("branch_id"), "category_en", tr.transfer_date.str[:7].rename("year_month")]).quantity_units.sum().rename("received")
bst = ms[ms.stock_location_type == "Branch"].rename(columns={"stock_location_id": "branch_id", "product_category": "category_en"})
bst = bst.set_index(["branch_id", "category_en", "year_month"]).join(sold).join(received).fillna({"sold": 0, "received": 0}).sort_index()
bst["change"] = bst.closing_units - bst.groupby(level=[0, 1]).closing_units.shift(1)
ok = bst.dropna(subset=["change"])
print("Correlation: stock change vs (received - sold):", round(ok.change.corr(ok.received - ok.sold), 3))
start, end = bst.groupby(level=[0, 1]).closing_units.first().sum(), bst.groupby(level=[0, 1]).closing_units.last().sum()
print(f"Branches: start {start:,} + received {int(bst.received.sum()):,} - sold {int(bst.sold.sum()):,} = {int(start + bst.received.sum() - bst.sold.sum()):,} | actual end: {end:,}")

Correlation: stock change vs (received - sold): 0.0
Branches: start 325,457 + received 12,677,158 - sold 3,398,655 = 9,603,960 | actual end: 315,812


In [96]:
# does stock follow sales over the years? and the "stockout risk" score
yr = bst.index.get_level_values("year_month").str[:4]
print("Avg stock per branch & category by year:", bst.closing_units.groupby(yr).mean().round(0).to_dict())
print("Avg sold per branch & category per month by year:", bst.sold.groupby(yr).mean().round(1).to_dict())
bst["months_of_cover"] = bst.closing_units / bst.sold.replace(0, float("nan"))
print("Median months of cover by year:", bst.months_of_cover.groupby(yr).median().round(1).to_dict())
print("Correlation: stockout risk score vs months of cover:", round(bst.stockout_risk_score.corr(bst.months_of_cover), 3))

Avg stock per branch & category by year: {'2018': 250.0, '2019': 251.0, '2020': 250.0, '2021': 251.0, '2022': 249.0, '2023': 253.0, '2024': 249.0, '2025': 250.0, '2026': 250.0}
Avg sold per branch & category per month by year: {'2018': 14.2, '2019': 18.1, '2020': 19.1, '2021': 21.9, '2022': 25.5, '2023': 29.8, '2024': 33.4, '2025': 38.2, '2026': 38.1}
Median months of cover by year: {'2018': 22.5, '2019': 17.6, '2020': 16.9, '2021': 14.9, '2022': 12.7, '2023': 11.5, '2024': 10.1, '2025': 8.9, '2026': 8.9}
Correlation: stockout risk score vs months of cover: 0.003


In [97]:
# stock share vs sales share by category (branches)
share = pd.DataFrame({"stock_%": bst.groupby(level="category_en").closing_units.mean(), "sales_%": sl_cat.groupby("category_en").quantity.sum()})
share = (share / share.sum() * 100).round(1)
share.sort_values("sales_%", ascending=False)

,stock_%,sales_%
category_en,,
Socks,6.4,26.8
Men tops,6.4,13.0
Men boxers,6.3,11.2
Women tops,6.4,10.0
Kids underwear,6.4,8.7
Men underwear,6.3,5.6
Girls pajamas,6.4,4.7
Boys pajamas,6.4,4.6
Women bras,6.3,4.3


In [98]:
# no build-up before the Nov-Dec season
total = ms.groupby("year_month").closing_units.sum()
print("Stock index by month of year (1 = normal):", (total.groupby(total.index.str[5:7]).mean() / total.mean()).round(2).to_dict())

Stock index by month of year (1 = normal): {'01': 0.99, '02': 1.0, '03': 0.98, '04': 1.0, '05': 0.99, '06': 0.99, '07': 1.01, '08': 1.03, '09': 1.0, '10': 1.0, '11': 1.01, '12': 1.0}


## Step 16: `inventory_transfers`

In [99]:
it = tables["inventory_transfers"].copy()
print(it.shape)
it.head()

(36500, 6)


,transfer_id,transfer_date,source_warehouse_id,destination_branch_id,transfer_status,transport_cost_EGP
0,TRF0000001,2025-02-10,WH09,BR021,Completed,450
1,TRF0000002,2024-10-28,WH05,BR003,Completed,155
2,TRF0000003,2021-11-29,WH04,BR041,Completed,1200
3,TRF0000004,2026-09-30,WH06,BR012,Completed,155
4,TRF0000005,2023-07-30,WH01,BR056,Completed,2500


In [100]:
print("Empty cells:", it.isna().sum().sum(), "| duplicate id:", it.transfer_id.duplicated().sum())
print("Columns in dictionary but NOT in file:", set(dd[dd.table_name == "inventory_transfers.csv"].field_name) - set(it.columns))
print("Status:", it.transfer_status.value_counts().to_dict())
print("Unknown warehouse codes:", (~it.source_warehouse_id.isin(w.warehouse_id)).sum(), "| unknown branch codes:", (~it.destination_branch_id.isin(b.branch_id)).sum())
it["units"] = it.transfer_id.map(tables["inventory_transfer_items"].groupby("transfer_id").quantity_units.sum())
it["year"] = it.transfer_date.str[:4]
per_branch = it.groupby("destination_branch_id").size()
print("Transfers per branch: min", per_branch.min(), "| max", per_branch.max())
print("Transfers without lines:", it.units.isna().sum(), "| total units:", int(it.units.sum()), f"= {it.units.sum() / si.quantity.sum():.1f}x the units sold")

Empty cells: 0 | duplicate id: 0
Columns in dictionary but NOT in file: set()
Status: {'Completed': 33640, 'In transit': 2145, 'Partially received': 715}
Unknown warehouse codes: 0 | unknown branch codes: 0
Transfers per branch: min 418 | max 495
Transfers without lines: 0 | total units: 12677158 = 3.7x the units sold


In [101]:
# units sent to branches vs units sold, per year
per_year = pd.DataFrame({"units_sent_M": it.groupby("year").units.sum() / 1e6,
                         "units_sold_M": si.groupby(si.order_date.str[:4]).quantity.sum() / 1e6})
per_year["sent_per_sold"] = per_year.units_sent_M / per_year.units_sold_M
per_year.round(2)

,units_sent_M,units_sold_M,sent_per_sold
2018,0.31,0.05,5.60
2019,1.58,0.28,5.69
2020,1.59,0.29,5.41
2021,1.57,0.34,4.68
2022,1.60,0.39,4.09
2023,1.58,0.46,3.45
2024,1.62,0.51,3.16
2025,1.59,0.59,2.72
2026,1.24,0.49,2.53


In [102]:
# status: "In transit" for years?
print(pd.crosstab(it.year, it.transfer_status, normalize="index").round(3), "\n")
open_old = it[(it.transfer_status != "Completed") & (it.year < "2026")]
print("Not completed and older than 2026:", len(open_old), "transfers |", int(open_old.units.sum()), "units | oldest:", open_old.transfer_date.min())

transfer_status  Completed  In transit  Partially received
year                                                      
2018                 0.932       0.048               0.020
2019                 0.922       0.061               0.018
2020                 0.919       0.058               0.023
2021                 0.916       0.064               0.020
2022                 0.922       0.057               0.021
2023                 0.919       0.063               0.019
2024                 0.922       0.059               0.019
2025                 0.928       0.053               0.019
2026                 0.925       0.058               0.017 

Not completed and older than 2026: 2593 transfers | 895857 units | oldest: 2018-10-20


In [103]:
# transport cost: does it depend on units, distance or year?
print("Cost values:", it.transport_cost_EGP.value_counts().sort_index().to_dict())
print("Correlation cost ~ units:", round(it.transport_cost_EGP.corr(it.units), 3))
geo = it.merge(w[["warehouse_id", "governorate_ar"]], left_on="source_warehouse_id", right_on="warehouse_id")
geo = geo.merge(b[["branch_id", "governorate_ar"]], left_on="destination_branch_id", right_on="branch_id", suffixes=("_wh", "_br"))
alex = geo[geo.governorate_ar_wh == "الإسكندرية"]
print("Avg cost from Alexandria to Aswan:", round(alex[alex.governorate_ar_br == "أسوان"].transport_cost_EGP.mean()),
      "| to Alexandria:", round(alex[alex.governorate_ar_br == "الإسكندرية"].transport_cost_EGP.mean()))
print("Avg cost by year:", it.groupby("year").transport_cost_EGP.mean().round(0).to_dict())

Cost values: {95: 5179, 155: 5047, 280: 5191, 450: 5202, 800: 5237, 1200: 5327, 2500: 5317}
Correlation cost ~ units: -0.004
Avg cost from Alexandria to Aswan: 700 | to Alexandria: 865
Avg cost by year: {'2018': 770.0, '2019': 791.0, '2020': 788.0, '2021': 803.0, '2022': 795.0, '2023': 789.0, '2024': 788.0, '2025': 800.0, '2026': 797.0}


## Step 17: `inventory_transfer_items`

In [104]:
iti = tables["inventory_transfer_items"]
print(iti.shape)
iti.head()

(109614, 4)


,transfer_line_id,transfer_id,sku_id,quantity_units
0,TRI00000001,TRF0000001,SKU000006,116
1,TRI00000002,TRF0000002,SKU000039,65
2,TRI00000003,TRF0000002,SKU000457,174
3,TRI00000004,TRF0000002,SKU000366,155
4,TRI00000005,TRF0000003,SKU000371,111


In [105]:
print("Empty cells:", iti.isna().sum().sum(), "| duplicate id:", iti.transfer_line_id.duplicated().sum(),
      "| lines without a transfer:", (~iti.transfer_id.isin(it.transfer_id)).sum(), "| unknown SKUs:", (~iti.sku_id.isin(v.sku_id)).sum())
print("Columns in dictionary but NOT in file:", set(dd[dd.table_name == "inventory_transfer_items.csv"].field_name) - set(iti.columns))
print("Lines per transfer:", iti.groupby("transfer_id").size().value_counts().sort_index().to_dict())
print("Units per line: min", iti.quantity_units.min(), "| max", iti.quantity_units.max(), "| avg", round(iti.quantity_units.mean(), 1), "| SKUs used:", iti.sku_id.nunique())

Empty cells: 0 | duplicate id: 0 | lines without a transfer: 0 | unknown SKUs: 0
Columns in dictionary but NOT in file: set()
Lines per transfer: {1: 7300, 2: 7290, 3: 7234, 4: 7348, 5: 7328}
Units per line: min 12 | max 219 | avg 115.7 | SKUs used: 695


In [106]:
# does each SKU get what it sells?
per_sku = pd.DataFrame({"sent": iti.groupby("sku_id").quantity_units.sum(), "sold": si.groupby("sku_id").quantity.sum()})
per_sku["sent_per_sold"] = per_sku.sent / per_sku.sold
print("Correlation sent ~ sold per SKU:", round(per_sku.sent.corr(per_sku.sold), 3))
print("Sent per SKU: min", per_sku.sent.min(), "max", per_sku.sent.max(), "|| sold per SKU: min", per_sku.sold.min(), "max", per_sku.sold.max())
per_sku = per_sku.join(v.set_index("sku_id").product_id).join(p.set_index("product_id").category_en, on="product_id")
pd.concat([per_sku.nlargest(2, "sold"), per_sku.nsmallest(2, "sold")])[["category_en", "sent", "sold", "sent_per_sold"]].round(1)

Correlation sent ~ sold per SKU: 0.021
Sent per SKU: min 13503 max 22741 || sold per SKU: min 957 max 11699


,category_en,sent,sold,sent_per_sold
sku_id,,,,
SKU000298,Men boxers,19552,11699,1.7
SKU000429,Men boxers,17111,11465,1.5
SKU000032,Socks,19282,957,20.1
SKU000279,Socks,16235,978,16.6


In [107]:
# share of units sent vs sold by category, and the season
print("Sock SKUs:", (per_sku.category_en == "Socks").sum(), "of", len(per_sku))
cat_share = per_sku.groupby("category_en")[["sent", "sold"]].sum()
print((cat_share / cat_share.sum() * 100).round(1).sort_values("sold", ascending=False).head(6), "\n")
sent_m = iti.merge(it[["transfer_id", "transfer_date"]], on="transfer_id")
by_m = sent_m.groupby(sent_m.transfer_date.str[5:7]).quantity_units.sum()
print("Units sent index by month of year (1 = normal):", (by_m / by_m.mean()).round(2).to_dict())

Sock SKUs: 380 of 695
                sent  sold
category_en               
Socks           54.7  26.8
Men tops         6.6  13.0
Men boxers       5.1  11.2
Women tops       5.0  10.0
Kids underwear   5.6   8.7
Men underwear    3.6   5.6 

Units sent index by month of year (1 = normal): {'01': 1.04, '02': 0.93, '03': 1.0, '04': 0.96, '05': 1.0, '06': 1.0, '07': 1.02, '08': 1.0, '09': 1.04, '10': 1.03, '11': 0.97, '12': 1.0}


## Step 18: `purchase_orders`

In [108]:
po = tables["purchase_orders"]
print(po.shape)
po.head()

(41500, 9)


,po_id,po_date,supplier_id,receiving_warehouse_id,po_status,currency,po_subtotal_EGP,payment_terms_days,po_purpose
0,PO0000001,2025-11-17,SUP013,WH09,Partial receipt,EGP,1704171.82,90,Store network stock
1,PO0000002,2024-09-21,SUP002,WH06,Received,EGP,617878.76,60,Seasonal procurement
2,PO0000003,2021-04-03,SUP021,WH02,Received,EGP,313993.79,45,Seasonal procurement
3,PO0000004,2019-09-12,SUP028,WH02,Partial receipt,EGP,19561.75,60,Store network stock
4,PO0000005,2022-04-07,SUP039,WH06,Received,EGP,583164.95,45,Seasonal procurement


In [109]:
print("Empty cells:", po.isna().sum().sum(), "| duplicate po_id:", po.po_id.duplicated().sum())
print("Columns in dictionary but NOT in file:", set(dd[dd.table_name == "purchase_orders.csv"].field_name) - set(po.columns))
print("Currency:", po.currency.value_counts().to_dict(), "| purpose:", po.po_purpose.value_counts().to_dict())
print("Payment terms:", po.payment_terms_days.value_counts().sort_index().to_dict())
po_lines = tables["purchase_order_items"]
diff = (po.set_index("po_id").po_subtotal_EGP - po_lines.groupby("po_id").line_cost_EGP.sum()).abs()
print("POs whose subtotal is not the sum of their lines:", (diff > 0.05).sum(), "\n")
pd.crosstab(po.po_date.str[:4], po.po_status, normalize="index").round(3)

Empty cells: 0 | duplicate po_id: 0
Columns in dictionary but NOT in file: set()
Currency: {'EGP': 41500} | purpose: {'Store network stock': 13941, 'Finished goods replenishment': 13855, 'Seasonal procurement': 13704}
Payment terms: {30: 10350, 45: 10351, 60: 10402, 90: 10397}
POs whose subtotal is not the sum of their lines: 0 



po_status,Partial receipt,Received
po_date,,
2018,0.315,0.685
2019,0.329,0.671
2020,0.336,0.664
2021,0.331,0.669
2022,0.326,0.674
2023,0.328,0.672
2024,0.328,0.672
2025,0.333,0.667
2026,0.337,0.663


In [110]:
# money: purchases vs sales per year (EGP millions)
orders = tables["sales_orders"]
money = pd.DataFrame({"purchases_M": po.groupby(po.po_date.str[:4]).po_subtotal_EGP.sum() / 1e6,
                      "sales_M": orders.groupby(orders.order_date.str[:4]).paid_total_EGP.sum() / 1e6})
money["purchases_per_sales"] = money.purchases_M / money.sales_M
print("Total purchases:", round(po.po_subtotal_EGP.sum() / 1e9, 2), "B EGP | total sales:", round(orders.paid_total_EGP.sum() / 1e9, 2), "B EGP")
money.round(1)

Total purchases: 29.15 B EGP | total sales: 0.96 B EGP


,purchases_M,sales_M,purchases_per_sales
2018,251.6,4.3,59.1
2019,1381.5,24.6,56.1
2020,1539.9,29.3,52.5
2021,1711.3,37.3,45.9
2022,2326.9,58.8,39.6
2023,3227.1,95.9,33.6
2024,5122.0,169.6,30.2
2025,6929.6,266.9,26.0
2026,6662.7,272.3,24.5


In [111]:
# units bought vs sold per year, and is "seasonal" buying really seasonal?
po_units_lines = po_lines.merge(po[["po_id", "po_date", "po_purpose"]], on="po_id")
bought_sold = pd.DataFrame({"bought_M": po_units_lines.groupby(po_units_lines.po_date.str[:4]).quantity.sum() / 1e6,
                            "sold_M": si.groupby(si.order_date.str[:4]).quantity.sum() / 1e6})
bought_sold["bought_per_sold"] = bought_sold.bought_M / bought_sold.sold_M
print("'Seasonal procurement' share of POs by month:", pd.crosstab(po.po_date.str[5:7], po.po_purpose, normalize="index")["Seasonal procurement"].round(2).to_dict())
by_month = po_units_lines.groupby(po_units_lines.po_date.str[5:7]).quantity.sum()
print("Units bought by month of year (1 = normal):", (by_month / by_month.mean()).round(2).to_dict(), "\n")
bought_sold.round(2)

'Seasonal procurement' share of POs by month: {'01': 0.33, '02': 0.33, '03': 0.33, '04': 0.34, '05': 0.34, '06': 0.33, '07': 0.32, '08': 0.31, '09': 0.34, '10': 0.33, '11': 0.33, '12': 0.33}
Units bought by month of year (1 = normal): {'01': 1.02, '02': 0.9, '03': 1.02, '04': 1.0, '05': 0.99, '06': 0.97, '07': 1.07, '08': 1.0, '09': 0.98, '10': 1.01, '11': 0.99, '12': 1.04} 



,bought_M,sold_M,bought_per_sold
2018,9.15,0.05,167.52
2019,42.43,0.28,152.71
2020,42.09,0.29,143.27
2021,42.68,0.34,126.86
2022,42.85,0.39,109.36
2023,42.12,0.46,92.03
2024,43.26,0.51,84.38
2025,41.74,0.59,71.20
2026,33.12,0.49,67.95


## Step 19: `purchase_order_items`

In [112]:
print(po_lines.shape)
po_lines.head()

(103334, 7)


,po_line_id,po_id,sku_id,quantity,unit_cost_EGP,line_cost_EGP,delivery_date
0,POL00000001,PO0000001,SKU000360,1653,97.79,161646.87,2025-12-06
1,POL00000002,PO0000001,SKU000186,2919,129.85,379032.15,2025-12-08
2,POL00000003,PO0000001,SKU000076,5276,121.50,641034.00,2026-01-10
3,POL00000004,PO0000001,SKU000592,2084,250.70,522458.80,2025-12-31
4,POL00000005,PO0000002,SKU000585,2977,92.87,276473.99,2024-11-07


In [113]:
print("Empty cells:", po_lines.isna().sum().sum(), "| duplicate id:", po_lines.po_line_id.duplicated().sum(),
      "| lines without a PO:", (~po_lines.po_id.isin(po.po_id)).sum(), "| unknown SKUs:", (~po_lines.sku_id.isin(v.sku_id)).sum())
print("Columns in dictionary but NOT in file:", set(dd[dd.table_name == "purchase_order_items.csv"].field_name) - set(po_lines.columns))
print("line_cost = quantity x unit_cost:", f"{((po_lines.quantity * po_lines.unit_cost_EGP - po_lines.line_cost_EGP).abs() < 0.01).mean():.0%}")
print("Lines per PO:", po_lines.groupby("po_id").size().value_counts().sort_index().to_dict())
print("Units per line: min", po_lines.quantity.min(), "| max", po_lines.quantity.max(), "| avg", round(po_lines.quantity.mean()))
lead = (pd.to_datetime(po_units_lines.delivery_date) - pd.to_datetime(po_units_lines.po_date)).dt.days
print("Days from PO to delivery: min", lead.min(), "| max", lead.max(), "| avg", round(lead.mean(), 1))

Empty cells: 0 | duplicate id: 0 | lines without a PO: 0 | unknown SKUs: 0
Columns in dictionary but NOT in file: set()
line_cost = quantity x unit_cost: 100%
Lines per PO: {1: 10473, 2: 10385, 3: 10477, 4: 10165}
Units per line: min 90 | max 6500 | avg 3285
Days from PO to delivery: min 0 | max 55 | avg 28.8


In [114]:
# is each SKU bought according to its sales?
buy = pd.DataFrame({"bought": po_lines.groupby("sku_id").quantity.sum(), "sold": si.groupby("sku_id").quantity.sum()})
buy["bought_per_sold"] = buy.bought / buy.sold
print("Correlation bought ~ sold per SKU:", round(buy.bought.corr(buy.sold), 3))
print("Bought per SKU: min", buy.bought.min(), "| max", buy.bought.max(), "|| bought / sold: min", round(buy.bought_per_sold.min()), "| max", round(buy.bought_per_sold.max()))
buy = buy.join(v.set_index("sku_id").product_id).join(p.set_index("product_id").category_en, on="product_id")
share = buy.groupby("category_en")[["bought", "sold"]].sum()
(share / share.sum() * 100).round(1).sort_values("sold", ascending=False).head(5)

Correlation bought ~ sold per SKU: 0.011
Bought per SKU: min 341864 | max 661785 || bought / sold: min 33 | max 584


,bought,sold
category_en,,
Socks,54.5,26.8
Men tops,6.5,13.0
Men boxers,4.9,11.2
Women tops,5.0,10.0
Kids underwear,5.9,8.7


In [115]:
# purchase cost vs the unit cost used in sales (same SKU, same year)
pc = po_units_lines.groupby(["sku_id", po_units_lines.po_date.str[:4]]).unit_cost_EGP.mean()
sc = si.groupby(["sku_id", si.order_date.str[:4]]).unit_cost_EGP.mean()
pc.index.names = sc.index.names = ["sku_id", "year"]
ratio = (pc / sc).dropna()
print("Purchase cost / sales cost, by year:", ratio.groupby(level="year").mean().round(2).to_dict())

Purchase cost / sales cost, by year: {'2018': 0.83, '2019': 0.83, '2020': 0.83, '2021': 0.83, '2022': 0.83, '2023': 0.83, '2024': 0.83, '2025': 0.83, '2026': 0.83}


## Step 20: `import_shipments`

In [116]:
imp = tables["import_shipments"].copy()
print(imp.shape)
imp.head()

(4200, 10)


,import_shipment_id,import_partner_id,shipment_date,port_name,container_type,commodity_class,customs_value_USD,freight_USD,duty_and_tax_USD,destination_warehouse_id
0,IS000001,IMP002,10/1/2019,Port Said Port,20FT,Accessories,117931,6246.19,19722.22,WH10
1,IS000002,IMP001,6/14/2024,Alexandria Port,20FT,Industrial machinery,115418,5895.23,21037.00,WH05
2,IS000003,IMP013,11/24/2023,Port Said Port,40FT,Packaging materials,174554,14098.40,7685.55,WH07
3,IS000004,IMP013,4/8/2021,Port Said Port,20FT,Cotton fabric,150560,9937.31,26248.02,WH12
4,IS000005,IMP004,6/22/2019,Port Said Port,20FT,Industrial machinery,118631,7744.07,16291.46,WH05


In [117]:
print("Empty cells:", imp.isna().sum().sum(), "| duplicate id:", imp.import_shipment_id.duplicated().sum())
print("Columns in dictionary but NOT in file:", set(dd[dd.table_name == "import_shipments.csv"].field_name) - set(imp.columns))
imp["date"] = pd.to_datetime(imp.shipment_date, format="%m/%d/%Y")
print("Dates:", imp.date.min().date(), "->", imp.date.max().date(), "| shipments per year:", imp.groupby(imp.date.dt.year).size().to_dict())
print(imp.commodity_class.value_counts().to_dict(), "|", imp.container_type.value_counts().to_dict(), "|", imp.port_name.value_counts().to_dict())
print("Totals (M USD): value", round(imp.customs_value_USD.sum() / 1e6, 1), "| freight", round(imp.freight_USD.sum() / 1e6, 1),
      "| duty & tax", round(imp.duty_and_tax_USD.sum() / 1e6, 1))
print("Value per year (M USD):", (imp.groupby(imp.date.dt.year).customs_value_USD.sum() / 1e6).round(1).to_dict())

Empty cells: 0 | duplicate id: 0
Columns in dictionary but NOT in file: set()
Dates: 2018-10-19 -> 2026-10-18 | shipments per year: {2018: 109, 2019: 543, 2020: 515, 2021: 523, 2022: 529, 2023: 547, 2024: 521, 2025: 508, 2026: 405}
{'Packaging materials': 1074, 'Cotton fabric': 1072, 'Accessories': 1032, 'Industrial machinery': 1022} | {'20FT': 2800, '40FT': 1400} | {'Sokhna Port': 1439, 'Port Said Port': 1391, 'Alexandria Port': 1370}
Totals (M USD): value 450.2 | freight 25.8 | duty & tax 44.9
Value per year (M USD): {2018: 12.2, 2019: 58.5, 2020: 54.7, 2021: 54.7, 2022: 58.0, 2023: 58.3, 2024: 55.5, 2025: 54.4, 2026: 43.9}


In [118]:
# what drives freight and duty? does a 40ft container cost more than a 20ft?
imp["freight_%"] = imp.freight_USD / imp.customs_value_USD * 100
imp["duty_%"] = imp.duty_and_tax_USD / imp.customs_value_USD * 100
print("Freight as % of value: min", round(imp["freight_%"].min(), 1), "| max", round(imp["freight_%"].max(), 1),
      "| correlation freight ~ value:", round(imp.freight_USD.corr(imp.customs_value_USD), 2))
print("Duty as % of value: min", round(imp["duty_%"].min(), 1), "| max", round(imp["duty_%"].max(), 1))
print("Average duty % by commodity:", imp.groupby("commodity_class")["duty_%"].mean().round(1).to_dict(), "\n")
imp.groupby("container_type")[["customs_value_USD", "freight_USD"]].mean().round(0)

Freight as % of value: min 2.5 | max 9.0 | correlation freight ~ value: 0.78
Duty as % of value: min 0.0 | max 20.0
Average duty % by commodity: {'Accessories': 10.0, 'Cotton fabric': 10.1, 'Industrial machinery': 9.9, 'Packaging materials': 10.1} 



,customs_value_USD,freight_USD
container_type,,
20FT,107182.0,6138.0
40FT,107218.0,6171.0


In [119]:
# how big are imports compared with all sales? (even at the lowest dollar rate of the period, about 15.5 EGP)
low_rate = 15.5
print("Imports at", low_rate, "EGP per USD:", round(imp.customs_value_USD.sum() * low_rate / 1e9, 1), "B EGP | all sales:",
      round(orders.paid_total_EGP.sum() / 1e9, 2), "B EGP")

Imports at 15.5 EGP per USD: 7.0 B EGP | all sales: 0.96 B EGP


## Step 21: `marketing_campaigns`

In [120]:
mc = tables["marketing_campaigns"].copy()
print(mc.shape)
mc.head()

(2200, 11)


,campaign_id,start_date,end_date,channel,target_audience,governorate_ar,budget_EGP,spend_EGP,impressions,clicks,orders_attributed
0,MKT00001,2026-10-14,2026-10-18,Meta Ads,Men,مطروح,161471,127561.63,5549655,275741,12326
1,MKT00002,2018-10-22,2018-11-10,Outdoor,Kids,بورسعيد,48867,34420.78,3086709,194598,1644
2,MKT00003,2021-04-14,2021-06-09,Meta Ads,Kids,مطروح,190681,129339.53,5447045,387692,18064
3,MKT00004,2020-05-27,2020-07-03,In-store,Women,الإسكندرية,225004,172433.63,9639206,765815,22649
4,MKT00005,2021-11-05,2021-12-30,Influencers,Men,مطروح,86691,69873.13,4825110,350546,29029


In [121]:
print("Empty cells:", mc.isna().sum().sum(), "| duplicate id:", mc.campaign_id.duplicated().sum())
print("Columns in dictionary but NOT in file:", set(dd[dd.table_name == "marketing_campaigns.csv"].field_name) - set(mc.columns))
mc["days"] = (pd.to_datetime(mc.end_date) - pd.to_datetime(mc.start_date)).dt.days + 1
print("Channels:", mc.channel.value_counts().to_dict())
print("Days: min", mc.days.min(), "| max", mc.days.max(), "| spend / budget: min", round((mc.spend_EGP / mc.budget_EGP).min(), 2),
      "| max", round((mc.spend_EGP / mc.budget_EGP).max(), 2))
print("Total spend:", round(mc.spend_EGP.sum() / 1e6, 1), "M EGP of budget", round(mc.budget_EGP.sum() / 1e6, 1), "M | correlation spend ~ days:", round(mc.spend_EGP.corr(mc.days), 3))
all_days = pd.date_range(orders.order_date.min(), orders.order_date.max())
running = sum(((all_days >= s) & (all_days <= e)).astype(int) for s, e in zip(pd.to_datetime(mc.start_date), pd.to_datetime(mc.end_date)))
print("Campaigns running on the same day: avg", round(running.mean(), 1), "| max", running.max())
print("Orders attributed:", mc.orders_attributed.sum(), "| real orders:", len(orders), f"| ratio {mc.orders_attributed.sum() / len(orders):.1f}x")
print("Orders attributed / clicks: min", round((mc.orders_attributed / mc.clicks).min(), 3), "| max", round((mc.orders_attributed / mc.clicks).max(), 3))

Empty cells: 0 | duplicate id: 0
Columns in dictionary but NOT in file: set()
Channels: {'Meta Ads': 387, 'Google Ads': 382, 'Influencers': 373, 'Outdoor': 367, 'In-store': 363, 'SMS / CRM': 328}
Days: min 1 | max 81 | spend / budget: min 0.6 | max 1.0
Total spend: 219.5 M EGP of budget 274.6 M | correlation spend ~ days: 0.003
Campaigns running on the same day: avg 33.4 | max 53
Orders attributed: 24106747 | real orders: 540000 | ratio 44.6x
Orders attributed / clicks: min 0.004 | max 0.095


In [122]:
# marketing spend vs sales per year, and the season
start_year = mc.start_date.str[:4]
mk_year = pd.DataFrame({"marketing_M": mc.groupby(start_year).spend_EGP.sum() / 1e6,
                        "sales_M": orders.groupby(orders.order_date.str[:4]).paid_total_EGP.sum() / 1e6})
mk_year["marketing_per_sales"] = mk_year.marketing_M / mk_year.sales_M
print("Share of spend in campaigns that start in Nov-Dec:", f"{mc[mc.start_date.str[5:7].isin(['11', '12'])].spend_EGP.sum() / mc.spend_EGP.sum():.1%}")
mk_year.round(2)

Share of spend in campaigns that start in Nov-Dec: 16.1%


,marketing_M,sales_M,marketing_per_sales
2018,4.59,4.26,1.08
2019,24.59,24.65,1.00
2020,23.64,29.31,0.81
2021,28.26,37.30,0.76
2022,30.22,58.80,0.51
2023,29.03,95.93,0.30
2024,25.72,169.61,0.15
2025,30.24,266.92,0.11
2026,23.23,272.28,0.09


In [123]:
# channels side by side
ch = mc.groupby("channel")[["spend_EGP", "impressions", "clicks", "orders_attributed"]].sum()
ch["CTR_%"] = ch.clicks / ch.impressions * 100
ch["cost_per_click"] = ch.spend_EGP / ch.clicks
ch["cost_per_attributed_order"] = ch.spend_EGP / ch.orders_attributed
ch[["CTR_%", "cost_per_click", "cost_per_attributed_order"]].round(2)

,CTR_%,cost_per_click,cost_per_attributed_order
channel,,,
Google Ads,4.36,0.46,9.36
In-store,4.58,0.43,8.42
Influencers,4.23,0.50,9.52
Meta Ads,4.67,0.42,8.39
Outdoor,4.48,0.47,9.98
SMS / CRM,4.40,0.48,9.22


In [124]:
# the first campaign vs the real orders in the same place and days
c1 = mc.iloc[0]
in_days = orders.order_date.between(c1.start_date, c1.end_date)
there = orders.branch_id.isin(b[b.governorate_ar == c1.governorate_ar].branch_id)
print(c1.campaign_id, c1.governorate_ar, c1.start_date, "->", c1.end_date, "| attributed:", c1.orders_attributed,
      "| real orders there:", (in_days & there).sum(), "| real orders in all branches:", in_days.sum())
# spend by governorate vs orders by governorate
gov_mk = pd.DataFrame({"spend_M": mc.groupby("governorate_ar").spend_EGP.sum() / 1e6,
                       "orders": orders.merge(b[["branch_id", "governorate_ar"]], on="branch_id").groupby("governorate_ar").size()})
gov_mk.loc[["القاهرة", "مطروح"]].round(1)

MKT00001 مطروح 2026-10-14 -> 2026-10-18 | attributed: 12326 | real orders there: 11 | real orders in all branches: 1238


,spend_M,orders
governorate_ar,,
القاهرة,10.7,141811
مطروح,10.2,6640


## Step 22: `payroll`

In [125]:
pr = tables["payroll"]
print(pr.shape)
pr.head()

(13210, 8)


,payroll_id,employee_id,pay_month,base_salary_EGP,overtime_EGP,bonus_EGP,deductions_EGP,net_paid_EGP
0,HR00000001,EMP00001,2023-06,4867.8,96.90,319.96,80.18,5204.48
1,HR00000002,EMP00001,2023-07,4867.8,232.56,458.66,99.56,5459.46
2,HR00000003,EMP00001,2023-08,4867.8,532.38,48.26,74.10,5374.34
3,HR00000004,EMP00001,2023-09,4867.8,105.64,233.70,170.24,5036.90
4,HR00000005,EMP00001,2023-10,4867.8,143.26,519.08,68.78,5461.36


In [126]:
print("Empty cells:", pr.isna().sum().sum(), "| duplicate id:", pr.payroll_id.duplicated().sum(),
      "| unknown employees:", (~pr.employee_id.isin(tables["employees"].employee_id)).sum())
print("Columns in dictionary but NOT in file:", set(dd[dd.table_name == "payroll.csv"].field_name) - set(pr.columns))
calc = pr.base_salary_EGP + pr.overtime_EGP + pr.bonus_EGP - pr.deductions_EGP
print("net = base + overtime + bonus - deductions:", f"{((calc - pr.net_paid_EGP).abs() < 0.01).mean():.0%}")
months_paid = pr.groupby("employee_id").size()
print("Months paid per employee: min", months_paid.min(), "| max", months_paid.max(), "| avg", round(months_paid.mean(), 1), "(the data covers 97 months)")
print("Employees paid in each year:", pr.groupby(pr.pay_month.str[:4]).employee_id.nunique().to_dict())
print("Employees paid in the last month", pr.pay_month.max(), ":", (pr.pay_month == pr.pay_month.max()).sum(), "of", len(tables["employees"]))

Empty cells: 0 | duplicate id: 0 | unknown employees: 0
Columns in dictionary but NOT in file: set()
net = base + overtime + bonus - deductions: 100%
Months paid per employee: min 9 | max 34 | avg 20.6 (the data covers 97 months)
Employees paid in each year: {'2018': 3, '2019': 62, '2020': 164, '2021': 264, '2022': 321, '2023': 322, '2024': 283, '2025': 185, '2026': 68}
Employees paid in the last month 2026-10 : 10 of 640


In [127]:
# raises: when does the base salary change, and by how much?
srt = pr.sort_values(["employee_id", "pay_month"])
prev = srt.groupby("employee_id").base_salary_EGP.shift()
raised = srt[prev.notna() & (srt.base_salary_EGP != prev)]
pct = (raised.base_salary_EGP / prev[raised.index] - 1) * 100
print("Month of every raise:", raised.pay_month.str[5:7].value_counts().to_dict())
print("Raise %: min", round(pct.min()), "| median", round(pct.median()), "| max", round(pct.max()))
print("Median raise % by year:", pct.groupby(raised.pay_month.str[:4]).median().round(0).to_dict())
print("Average monthly net by year:", pr.groupby(pr.pay_month.str[:4]).net_paid_EGP.mean().round(0).to_dict())

Month of every raise: {'01': 1032}
Raise %: min 11 | median 39 | max 55
Median raise % by year: {'2019': 14.0, '2020': 12.0, '2021': 11.0, '2022': 35.0, '2023': 41.0, '2024': 55.0, '2025': 39.0, '2026': 22.0}
Average monthly net by year: {'2018': 1974.0, '2019': 1802.0, '2020': 1989.0, '2021': 2168.0, '2022': 2972.0, '2023': 4261.0, '2024': 6564.0, '2025': 9019.0, '2026': 10635.0}


In [128]:
# does payroll agree with the employees table? (2026), and how much is missing?
p26 = pr[pr.pay_month.str[:4] == "2026"].groupby("employee_id").base_salary_EGP.last()
cmp26 = p26.to_frame("payroll_base_2026").join(tables["employees"].set_index("employee_id").salary_base_EGP_2026)
print("Employees paid in 2026:", len(cmp26), "| correlation with employees table:", round(cmp26.payroll_base_2026.corr(cmp26.salary_base_EGP_2026), 2))
pr25 = pr[pr.pay_month.str[:4] == "2025"]
print("Payroll recorded in 2025:", round(pr25.net_paid_EGP.sum() / 1e6, 1), "M EGP | if all 640 were paid at the 2025 average:",
      round(640 * 12 * pr25.net_paid_EGP.mean() / 1e6, 1), "M EGP")
cmp26.head()

Employees paid in 2026: 68 | correlation with employees table: -0.08
Payroll recorded in 2025: 13.8 M EGP | if all 640 were paid at the 2025 average: 69.3 M EGP


,payroll_base_2026,salary_base_EGP_2026
employee_id,,
EMP00005,10821.0,16100
EMP00006,8330.0,13800
EMP00034,8936.0,9500
EMP00060,7233.0,8600
EMP00065,9638.0,13700


## Step 23: `operating_expenses`

In [129]:
ox = tables["operating_expenses"]
print(ox.shape)
ox.head()

(47000, 7)


,expense_id,expense_date,branch_or_department_id,expense_category,amount_EGP,payment_method,vendor_name
0,EXP0000001,2018-11-23,WH01,Travel,540.68,Card,Vendor 051
1,EXP0000002,2025-03-18,BR036,General administration,2257.46,Cash,Vendor 197
2,EXP0000003,2025-07-16,BR043,General administration,3602.26,Cash,Vendor 070
3,EXP0000004,2023-10-16,BR075,Utilities,3047.98,Cash,Vendor 195
4,EXP0000005,2018-11-05,BR077,Travel,1192.94,Cash,Vendor 040


In [130]:
print("Empty cells:", ox.isna().sum().sum(), "| duplicate id:", ox.expense_id.duplicated().sum())
print("Columns in dictionary but NOT in file:", set(dd[dd.table_name == "operating_expenses.csv"].field_name) - set(ox.columns))
print("Where:", ox.branch_or_department_id.str[:2].value_counts().to_dict(), "| vendors:", ox.vendor_name.nunique(), "| payment:", ox.payment_method.value_counts().to_dict())
print("Total:", round(ox.amount_EGP.sum() / 1e6, 1), "M EGP | average amount by year:", ox.groupby(ox.expense_date.str[:4]).amount_EGP.mean().round(0).to_dict(), "\n")
pd.DataFrame({"rows": ox.expense_category.value_counts(), "avg_amount": ox.groupby("expense_category").amount_EGP.mean(),
              "total_M": ox.groupby("expense_category").amount_EGP.sum() / 1e6}).round(1)

Empty cells: 0 | duplicate id: 0
Columns in dictionary but NOT in file: set()
Where: {'BR': 37592, 'WH': 9408} | vendors: 250 | payment: {'Bank transfer': 15926, 'Cash': 15593, 'Card': 15481}
Total: 135.6 M EGP | average amount by year: {'2018': 920.0, '2019': 1075.0, '2020': 1221.0, '2021': 1336.0, '2022': 1833.0, '2023': 2563.0, '2024': 4002.0, '2025': 5528.0, '2026': 6847.0} 



,rows,avg_amount,total_M
expense_category,,,
General administration,4652,2375.7,11.1
IT & software,4609,2353.2,10.8
Insurance,4646,2401.4,11.2
Logistics,4742,2393.2,11.3
Maintenance,4715,2358.0,11.1
Marketing,4856,2421.8,11.8
Rent,4692,4896.3,23.0
Supplies,4645,2344.2,10.9
Travel,4779,2417.7,11.6


In [131]:
# rent: is it paid every month, and does it depend on the place?
rent = ox[(ox.expense_category == "Rent") & ox.branch_or_department_id.str.startswith("BR")]
rent_months = rent[rent.expense_date.between("2019-01-01", "2025-12-31")].groupby("branch_or_department_id").expense_date.apply(lambda s: s.str[:7].nunique())
print("Months with rent per branch, 2019-2025 (of 84): min", rent_months.min(), "| max", rent_months.max(), "| avg", round(rent_months.mean(), 1))
rent_total = rent.groupby("branch_or_department_id").amount_EGP.sum()
print("Total rent per branch: in a mall", round(rent_total[rent_total.index.isin(mall.branch_id)].mean()),
      "| not in a mall", round(rent_total[~rent_total.index.isin(mall.branch_id)].mean()), "\n")
pd.crosstab(ox.expense_category, ox.payment_method, normalize="index").round(2).loc[["Rent", "Utilities", "Travel"]]

Months with rent per branch, 2019-2025 (of 84): min 23 | max 41 | avg 32.6
Total rent per branch: in a mall 243007 | not in a mall 230519 



payment_method,Bank transfer,Card,Cash
expense_category,,,
Rent,0.34,0.34,0.33
Utilities,0.34,0.32,0.34
Travel,0.34,0.33,0.33


In [132]:
# branch expenses vs branch sales, marketing in two places, and expenses as a share of sales
spent = ox[ox.branch_or_department_id.str.startswith("BR")].groupby("branch_or_department_id").amount_EGP.sum()
net_sales = si.groupby("branch_id").net_line_EGP.sum()
print("Expenses per branch: min", round(spent.min() / 1e6, 2), "M | max", round(spent.max() / 1e6, 2), "M | correlation with branch sales:", round(spent.corr(net_sales), 2))
print("'Marketing' in expenses:", round(ox[ox.expense_category == "Marketing"].amount_EGP.sum() / 1e6, 1), "M | marketing_campaigns:", round(mc.spend_EGP.sum() / 1e6, 1), "M")
(ox.groupby(ox.expense_date.str[:4]).amount_EGP.sum() / orders.groupby(orders.order_date.str[:4]).net_before_tax_EGP.sum()).round(3).to_frame("expenses / net sales")

Expenses per branch: min 1.05 M | max 1.53 M | correlation with branch sales: -0.11
'Marketing' in expenses: 11.8 M | marketing_campaigns: 219.5 M


,expenses / net sales
expense_date,
2018,0.305
2019,0.291
2020,0.281
2021,0.240
2022,0.210
2023,0.182
2024,0.157
2025,0.136
2026,0.133


## Step 24: `data_dictionary`

In [133]:
# read the files again (first rows only), because earlier steps added helper columns to some tables
raw = {f[:-4]: pd.read_csv(os.path.join(DATA, f), nrows=5) for f in sorted(os.listdir(DATA)) if f.endswith(".csv")}
print(dd.shape)
print("Tables described:", dd.table_name.nunique(), "| files in the folder:", len(raw))
just_name = (dd.description.str.replace(" ", "_").str.lower() == dd.field_name.str.lower()).mean()
print("Descriptions that only repeat the column name:", f"{just_name:.0%}")
for t, df in raw.items():
    listed = set(dd[dd.table_name == t + ".csv"].field_name)
    if t != "data_dictionary" and listed != set(df.columns):
        print(t, "| in dictionary only:", listed - set(df.columns), "| in file only:", set(df.columns) - listed)
dd.head(8)

(169, 3)
Tables described: 23 | files in the folder: 24
Descriptions that only repeat the column name: 100%
branches | in dictionary only: {'source_url'} | in file only: set()
customers | in dictionary only: {'loyalty_tier'} | in file only: set()


,table_name,field_name,description
0,branches.csv,branch_id,branch id
1,branches.csv,branch_name_ar,branch name ar
2,branches.csv,governorate_ar,governorate ar
3,branches.csv,city_ar,city ar
4,branches.csv,address_ar,address ar
5,branches.csv,retail_location_type,retail location type
6,branches.csv,opening_date,opening date
7,branches.csv,source_url,source url


In [134]:
# date formats are not the same in all tables
import re
for t, df in raw.items():
    for col in df.columns:
        if "date" in col:
            first = str(df[col].iloc[0])
            print(f"{t:26s} {col:15s} {'YYYY-MM-DD' if re.match(r'\d{4}-', first) else 'M/D/YYYY':11s} e.g. {first}")

branches                   opening_date    M/D/YYYY    e.g. 9/14/2019
customers                  joined_date     M/D/YYYY    e.g. 10/19/2018
delivery_shipments         ship_date       YYYY-MM-DD  e.g. 2018-10-20
delivery_shipments         delivery_date   YYYY-MM-DD  e.g. 2018-10-23
employees                  hire_date       M/D/YYYY    e.g. 10/19/2022
import_shipments           shipment_date   M/D/YYYY    e.g. 10/1/2019
inventory_transfers        transfer_date   YYYY-MM-DD  e.g. 2025-02-10
marketing_campaigns        start_date      YYYY-MM-DD  e.g. 2026-10-14
marketing_campaigns        end_date        YYYY-MM-DD  e.g. 2026-10-18
operating_expenses         expense_date    YYYY-MM-DD  e.g. 2018-11-23
purchase_order_items       delivery_date   YYYY-MM-DD  e.g. 2025-12-06
purchase_orders            po_date         YYYY-MM-DD  e.g. 2025-11-17
sales_order_items          order_date      YYYY-MM-DD  e.g. 2018-10-19
sales_orders               order_date      YYYY-MM-DD  e.g. 2018-10-19
sales_pa

## Step 25: Public catalog (`01_CATALOG`)

In [135]:
pub = pd.read_csv(os.path.join(os.path.dirname(DATA), "01_CATALOG", "PRODUCT_CATALOG_PUBLIC_REFERENCE.csv"))
print(pub.shape)
pub.head()

(238, 4)


,product_id,product_name_ar,category,listed_price_EGP
0,PR0001,3 * شراب رجالي كلاسيك-لايكرا 1,Socks,290
1,PR0002,3 * شراب رجالي كلاسيك-لايكرا 2,Socks,290
2,PR0003,3 * شراب رجالي كلاسيك-لايكرا 3,Socks,290
3,PR0004,3 * شراب رجالي كلاسيك-لايكرا 4,Socks,290
4,PR0005,3 * شراب رجالي كلاسيك-لايكرا 5,Socks,290


In [136]:
print("Empty cells:", pub.isna().sum().sum(), "| duplicate ids:", pub.product_id.duplicated().sum())
both = pub.merge(p, on="product_id", suffixes=("_pub", ""))
print("Products also in products_catalog:", len(both), "| same name:", f"{(both.product_name_ar_pub == both.product_name_ar).mean():.0%}",
      "| same price:", f"{(both.listed_price_EGP_pub == both.listed_price_EGP).mean():.0%}")
extra = pub[~pub.product_id.isin(p.product_id)]
print("Extra products:", len(extra), "| ids", extra.product_id.min(), "->", extra.product_id.max(), "| sold in the data:", extra.product_id.isin(si.product_id).sum())
print("Category names of the extra products:", extra.category.value_counts().to_dict())
extra.head()

Empty cells: 0 | duplicate ids: 0
Products also in products_catalog: 139 | same name: 100% | same price: 100%
Extra products: 99 | ids PUB0001 -> PUB0099 | sold in the data: 0
Category names of the extra products: {'رجالي': 52, 'بناتي': 26, 'أطفال': 12, 'ولادي': 5, 'حريمي': 3, 'شرابات': 1}


,product_id,product_name_ar,category,listed_price_EGP
139,PUB0001,بوكسر تيربو أطفال 2,أطفال,85
140,PUB0002,بوكسر تيربو أطفال 3,أطفال,85
141,PUB0003,بوكسر تيربو أطفال 4,أطفال,85
142,PUB0004,بوكسر تيربو أطفال 5,أطفال,85
143,PUB0005,بوكسر تيربو أطفال 6,أطفال,85
